In [298]:
# принудительно перезагрузить модуль
import importlib
import df_preprocessing

importlib.reload(df_preprocessing)

# теперь импортируем функции заново
from df_preprocessing import add_original_to_extended

In [299]:
import os
import re


from collections import defaultdict
import copy

import numpy as np
import pandas as pd
from IPython.display import display
import networkx as nx
import seaborn as sns
import matplotlib.pyplot as plt

from df_preprocessing import (
    load_repair_parts,
    extract_model_case_insensitive,
    extract_articles,
    add_original_to_extended,
    find_all_analogs,
    merge_parts_df1,
    merge_parts_df2,
    prepare_sales,
    normalize,
    normalize_analog_lists,
    
)

In [300]:
file_csv = r'Запчасти списанные в ремонт.csv'
file_excel = r'Запчасти списанные в ремонт.xlsx'
data = load_repair_parts(file_csv, file_excel)

data = data[data['Номенклатура.Артикул'].notna()]
data = data.drop(columns=['Unnamed: 1', 'Unnamed: 5', 'Unnamed: 6'])
data = data.iloc[:-1].reset_index(drop=True)
data = data[~(data['Номенклатура.Оригинальный номер'].isna() & data['Номенклатура.Артикул'].isna())]

conditions1 = [
    data['Машина'].str.contains('ножничный', case=False, na=False),
    data['Машина'].str.contains('коленчатый', case=False, na=False),
    data['Машина'].str.contains('телескопический', case=False, na=False),
    data['Машина'].str.contains('мачтовый', case=False, na=False)
]

choices1 = ['ножничный', 'коленчатый', 'телескопический', 'мачтовый']

data['Тип подъемника'] = np.select(conditions1, choices1, default='другое')

conditions2 = [

    data['Машина'].str.contains('электрический', case=False, na=False),
    data['Машина'].str.contains('дизельный', case=False, na=False),
]

choices2 = ['Электрический', 'Дизельный']

data['Тип двигателя'] = np.select(conditions2, choices2, default='другое')

data = data.loc[~(data['Тип подъемника'] == 'другое')]

data = data[~(data['Машина'].str.startswith("Телескопический погрузчик"))]

data.loc[data['Машина'] == 'Телескопический подъемник Zoomlion ZT72J-V', 'Тип двигателя'] = 'Дизельный'
data.loc[data['Машина'] == 'Телескопический подъемник Zoomlion ZT30J', 'Тип двигателя'] = 'Дизельный'

data['Дата'] = pd.to_datetime(data['Дата'], format='%d.%m.%Y %H:%M:%S')
data['Год'] = data['Дата'].dt.year
data['Квартал'] = data['Дата'].dt.quarter
data['Лот.CRM год выпуска'] = data['Лот.CRM год выпуска'].apply(lambda x: int(x.replace(',', '')) if isinstance(x, str) else x)
data['Средняя наработка (лет)'] = data['Год'] - data['Лот.CRM год выпуска']

col = data.pop("Квартал")  
data.insert(2, "Квартал", col)

data['Номера машин'] = data.apply(lambda row: extract_model_case_insensitive(row['Машина'], row['Машина.Бренд']), axis=1)

Файл успешно загружен из CSV: Запчасти списанные в ремонт.csv


In [301]:
df1 = copy.deepcopy(data)

In [302]:
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'JCPT1412HA','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'AMWP11.5-8100','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'] == 'ZS1212','Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура.Артикул'] == 'DIFA438201', 'Номенклатура.Оригинальный номер расширенный'] = '4382-01'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 32925682', 'Номенклатура.Оригинальный номер расширенный'] = '32/925682 AF26656'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 32925682', 'Номенклатура.Оригинальный номер'] = '32/925682'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Артикул'] = '85000159'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Оригинальный номер'] = '85000159'
df1.loc[df1['Номенклатура'] == 'НПУ 00007940', 'Номенклатура.Оригинальный номер расширенный'] = '85000159'
df1.loc[df1['Номенклатура'] == 'Цилиндр поворота платформы 1010201914', 'Номенклатура.Оригинальный номер'] = '1010201914'
df1.loc[df1['Номенклатура'] == 'Трос аварийного опускания Haulotte Compact', 'Номенклатура.Оригинальный номер'] = '2420314420'
df1.loc[df1['Номенклатура'] == 'Трос аварийного опускания Haulotte Compact', 'Номенклатура.Оригинальный номер расширенный'] = '2326009100'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный 90031461A', 'Номенклатура.Оригинальный номер расширенный'] = '43121'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный C-7926', 'Номенклатура.Оригинальный номер расширенный'] = '1009900560 1009901968 15056-32432 15521-32439 16098-31291 1C01032432 1C020-32430 1C020-32432 1C020-32433 1C020-32434 1C020-32434-P 1C02032434 279809 4042216 582018366 807180 HH1C0-32430 HH1C032430 HLX-6995 LF3376 P550318'
df1.loc[df1['Номенклатура'] == 'Датчик 203150000055', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Фильтр топливный Yanmar DF-5543', 'Номенклатура.Оригинальный номер расширенный'] = '201990003015 60356164 7029012 A977950 MIU802421 SF-52010 SF52010 SK48584 SN25127 XJAU01355 XJAU01515 YM129A0055730'
df1.loc[df1['Номенклатура'] == 'Клапан балансировочный поворотного редуктора корзины 1019901661', 'Номенклатура.Оригинальный номер расширенный'] = '1010201471 1010202355 1010202356 ls2-009-180qmajc'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический в сборе 00001871', 'Номенклатура.Оригинальный номер расширенный'] = '00004696 0160D010BN3HC SH75028'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный A8506S', 'Номенклатура.Оригинальный номер расширенный'] = '1000100310 P827653 RF3091AB ST40706'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный SFC7945002', 'Номенклатура.Оригинальный номер расширенный'] = '1010601542 SFC-79450-02 SFC7945002 SK3170 ST20094'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический CS05030P10A', 'Номенклатура.Оригинальный номер расширенный'] = '202080000036 SH56560 SP-06X10G3 SPF-53 SPF-538-100 stx202027'
df1.loc[df1['Номенклатура'] == 'Элемент фильтра гидравлического 00004416', 'Номенклатура.Оригинальный номер расширенный'] = '00003113 00004416'
df1.loc[df1['Номенклатура'] == 'Стартер Kubota 2403 TT15790', 'Номенклатура.Оригинальный номер расширенный'] = '17123-6301-6 17123-63017 1712363016 1712363017 CPK00215'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный FS36209', 'Номенклатура.Оригинальный номер расширенный'] = '1000000876 201990000029'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный ГО BA16-22CRT 10003966A (Kubota)', 'Номенклатура.Оригинальный номер расширенный'] = '1g311-43380 1g31143380 8972133810 FF5468 PF9873 SK3686 SN21586'
df1.loc[df1['Номенклатура'] == 'Фильтр топливный GB6245', 'Номенклатура.Оригинальный номер расширенный'] = '1000700908 PL420 ST21350'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный MANN-W 814/80', 'Номенклатура.Оригинальный номер расширенный'] = '11923630 199100323 2151740 2151740 749613 HH164-32430'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный DIFA 43121', 'Номенклатура.Оригинальный номер расширенный'] = '90029290 90031461A AF26656 P608533 P609490 ST40110 Z32540'
df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический R0110A10NHA', 'Номенклатура.Оригинальный номер расширенный'] = 'LH0110R010BN/HC DH3723 RHR110G10B3/AB1 P170604'
df1.loc[df1['Номенклатура'] == 'ЭБУ 203020000001', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Зарядное устройство 2029900000311', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Насос гидравлический 81013GT', 'Номенклатура.Оригинальный номер расширенный'] = np.nan
df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внутренний) 4382-01', 'Номенклатура.Оригинальный номер расширенный'] = 'DIFA 4382-01'
df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внутренний) 4382-01', 'Номенклатура.Артикул'] = 'DIFA 4382-01'
df1.loc[df1['Номенклатура'] == 'Замок зажигания 90173', 'Номенклатура.Оригинальный номер расширенный'] = '84830'
df1.loc[df1['Номенклатура'] == 'Стартер 1002415445С', 'Номенклатура.Оригинальный номер расширенный'] = '1003692596B 1009806744'

df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический RHR110G10B3/AB1', 'Номенклатура.Оригинальный номер расширенный'] = '(LH)0110A10BN/HA(HC) 0110R010BN3HC KFSF-0110-R-010-N R0110 R0110A010NHA R0110A10NHA'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный Z32540', 'Номенклатура.Оригинальный номер'] = '90029290'

df1.loc[df1['Номенклатура'] == 'Фильтрующий элемент очистки воздуха (внешний) 4382', 'Номенклатура.Оригинальный номер расширенный'] = np.nan

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный внутренний P829333', 'Номенклатура.Оригинальный номер расширенный'] = '216070000008 4383-01'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный внешний P828889', 'Номенклатура.Оригинальный номер расширенный'] = '216070000004 4383'

df1.loc[df1['Номенклатура'] == 'Фильтр масляный C1110', 'Номенклатура.Оригинальный номер расширенный'] = '90024289 SO228 C-1405 LS32665 01174416'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный комплект p828889', 'Номенклатура.Оригинальный номер расширенный'] = '216070000004 4383'
df1.loc[df1['Номенклатура'] == 'Фильтр воздушный комплект p828889', 'Номенклатура.Оригинальный номер'] = '00006753'

df1.loc[df1['Номенклатура'] == 'Фильтр гидравлический D0110A10NHA', 'Номенклатура.Оригинальный номер расширенный'] = '00004697 D0110A10NHA LH0110D010BN3HC'

df1.loc[df1['Номенклатура'] == 'Фильтр воздушный DIFA 43121-01', 'Номенклатура.Артикул'] = 'DIFA 43121-01'

df1.loc[df1['Номенклатура'] == 'Фара 00006753A/00005190A', 'Номенклатура.Артикул'] = '00006753'
df1.loc[df1['Номенклатура'] == 'Фара 00006753A/00005190A', 'Номенклатура'] = 'Фара головного света 00006753'
df1.loc[df1['Номенклатура'] == 'Фильтр масляный C-1049', 'Номенклатура.Артикул'] = 'C1049'

cols = [
    'Номенклатура.Артикул',
    'Номенклатура.Оригинальный номер',
    'Номенклатура.Оригинальный номер расширенный'
]

df1[cols] = df1[cols].replace(r'DIFA\s*', '', regex=True)

df1[cols] = df1[cols].replace(r'\s+', ' ', regex=True).apply(lambda x: x.str.strip())

df1.loc[df1['Номенклатура.Артикул'] == '438201', 'Номенклатура.Артикул'] = '4382-01'

mask_plus = (
    df1['Номенклатура'].str.contains(r'\+', na=False) &
    ~df1['Номенклатура'].str.contains(r'^(?:Колесо|РВД|[Оо]богреватель|Контроллер|Deutz|Выключатель|Батарея|Рукав|Фильтр воздушный к-кт \(внутр\.\+внешн\.\) 1351230502)', na=False) # Учесть фильтр последний (т.е найти его аналоги)
)

mask_st_units = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111/40110',
    case=False,
    na=False
)

mask_brackets = df1['Номенклатура'].str.contains(
    r'(95*165*340/70*90*335,',
    case=False,
    na=False,
    regex=False
)

mask_kt = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный AF26656 к-т',
    case=False,
    na=False,
)

mask_AB = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40131AB',
    case=False,
    na=False,
)

mask_hid = df1['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111ST40110',
    case=False,
    na=False,
)

complects = df1[mask_plus | mask_st_units | mask_brackets | mask_kt | mask_AB | mask_hid]
complects.loc[complects['Номенклатура'] == 'Фильтр воздушный AF26656 к-т', 'Номенклатура'] = 'Фильтр воздушный к-т AF26656+AF26655'
complects.loc[complects['Номенклатура'] == 'Фильтр воздушный ST40131AB', 'Номенклатура'] = 'Фильтр воздушный (95*165*340/70*90*335, сдвоенный) kw1634'

df1 = df1[~df1.index.isin(complects.index)]


In [303]:
df1.loc[df1['Номенклатура.Оригинальный номер расширенный'].isin(['438201'])]

,Дата,Год,Квартал,Месяц,Номенклатура,Номенклатура.Артикул,Номенклатура.Оригинальный номер,Номенклатура.Оригинальный номер расширенный,Номенклатура.Тип техники,Машина,...,Машина.Серия техники,Лот.CRM год выпуска,Серийный номер,Лот.CRM наработка,Документ.Склад,Количество,Тип подъемника,Тип двигателя,Средняя наработка (лет),Номера машин


In [304]:
matches = {
    '43121': '90029290 90031461A AF26656 P608533 P609490 ST40110 Z32540', # внешниий
    '43121-01': '90031459 P600975 Z32603', # внутренний
    '4382': '1000100310 P827653', # внешниий
    '4382-01': '1000100311 P829332', # внутренний
    'kw1634': '201020003059', # внешниий
    'kw1634in': '201020003058', # внутренний
    'AF26656': '32/925682 ST40110', # внешниий
    'AF26655': '32/925683 ST40111', # внутренний
    'ST40110': '90031461 AF26656', # внешний
    'ST40111': '90031459 AF26655', # внутренний
    '4383': '216070000004 P828889', # внешниий
    '4383-01': '216070000008 P829333', # внутренний
    'AF2555': '1000100310 6666375 B222100000593', # внешниий
    'AF25484': '1000100311 6666376 B222100000591', # внутренний
    'AP31642K': '216070000004', # внешний
    'AP31643K': '216070000008', # внутренний. Возможно аналог DIFA 4383-01
    '6666375': '1000100310', # внешниий
    '6666376': '1000100311', # внутренний
    '1000100310': '4382', # внешний
    '1000100311': '4382-01', # внутренний
    '43123': '43123', # внешний
    '43123-01': '43123-01', # внутренний
    '4338': '4338', # внешний
    '4338-01': '4338-01', # внутренний
    'ST40722(1)': '4338',  # внутренний
    'ST40722(2)': '4338-01'  # внутренний
}
	

complects['Номенклатура.Артикул'] = complects['Номенклатура'].apply(lambda x: extract_articles(x, matches.keys()))

df_exploded = complects.explode('Номенклатура.Артикул').reset_index(drop=True)

df_exploded['Номенклатура.Оригинальный номер'] = df_exploded['Номенклатура.Артикул'].map(lambda x: matches[x].split()[0] if x in matches else None)
df_exploded['Номенклатура.Оригинальный номер расширенный'] = df_exploded['Номенклатура.Артикул'].map(matches)

assert (df_exploded.shape[0] == complects.shape[0] * 2)

df1 = pd.concat([df1, df_exploded], ignore_index=True)

df1 = df1.sort_values(by='Дата', ignore_index=True)

In [305]:
cols = [
    'Номенклатура.Артикул',
    'Номенклатура.Оригинальный номер',
    'Номенклатура.Оригинальный номер расширенный'
]

for i in cols:
    df1[i] = df1[i].apply(normalize)

df1["Номенклатура.Оригинальный номер расширенный"] = df1.apply(add_original_to_extended, axis=1) 

df1['Аналоги'] = (
    df1['Номенклатура.Оригинальный номер расширенный']
    .fillna('')         
    .str.upper()
    .str.split()
)

graph = defaultdict(set)

for _, row in df1.iterrows():
    part = row['Номенклатура.Артикул']
    
    for analog in row['Аналоги']:
        if analog != part: 
            graph[part].add(analog)
            graph[analog].add(part)


df1['all_analogs'] = df1['Номенклатура.Артикул'].apply(lambda x: find_all_analogs(x, graph))
df1 = df1.drop(columns='Аналоги')

group_mapping = {}
group_counter = 1

for group_tuple in df1['all_analogs'].unique():
    group_mapping[group_tuple] = group_counter
    group_counter += 1

df1['Номер группы'] = df1['all_analogs'].map(group_mapping)

In [306]:
# df1.to_excel('text.xlsx', index=False)

In [307]:
# TODO Заняться очистой групп, найти несоответствия между номенклатурой и артикулами

In [308]:
df_quarter = merge_parts_df1(df1)

In [309]:
data2 = load_repair_parts('Остатки и обороты.csv', 'Остатки и обороты.xlsx', skiprows=8)
data2 = data2.iloc[:-2].reset_index(drop=True)
data2 = data2.rename(columns={'Артикул ': 'Артикул'})

period_mask = data2['Номенклатура'].str.contains(
    r'\d{4} г\.|\d квартал \d{4} г\.|Январь|Февраль|Март|Апрель|Май|Июнь|Июль|Август|Сентябрь|Октябрь|Ноябрь|Декабрь',
    na=False
)

data2['year'] = data2['Номенклатура'].str.extract(r'\b(20[2-9]\d)\b')
data2['quarter'] = data2['Номенклатура'].str.extract(r'(\d квартал)')
data2['month'] = data2['Номенклатура'].str.extract(
    r'(Январь|Февраль|Март|Апрель|Май|Июнь|Июль|Август|Сентябрь|Октябрь|Ноябрь|Декабрь)'
)
data2[['year','quarter','month']] = data2[['year','quarter','month']].ffill()
data2 = data2[~period_mask]

data2['Квартал'] = data2['quarter'].str.extract(r'(\d)').astype(int)

month_map = {
    'Январь': 1, 'Февраль': 2, 'Март': 3, 'Апрель': 4, 'Май': 5, 'Июнь': 6,
    'Июль': 7, 'Август': 8, 'Сентябрь': 9, 'Октябрь': 10, 'Ноябрь': 11, 'Декабрь': 12
}
data2['Месяц'] = data2['month'].map(month_map).astype(int)
data2['Год'] = data2['year']
data2.drop(columns=['Unnamed: 0', 'month', 'quarter', 'year'], inplace=True)
data2 = data2.loc[(data2['Артикул'].notna()) | (data2['Оригинальный номер'].notna())]

for col in ["Расход", "Приход", "Конечный остаток"]:
    data2[col] = (
        data2[col]
        .str.replace(',', '', regex=False)
        .astype(float)
    )
# data2.to_excel('test2.xlsx', index=False)

Файл успешно загружен из CSV: Остатки и обороты.csv


In [310]:
df2 = copy.deepcopy(data2)

In [311]:
cols = [
    'Артикул',
    'Оригинальный номер',
]

df2[cols] = df2[cols].replace(r'DIFA\s*', '', regex=True)

df2[cols] = df2[cols].replace(r'\s+', ' ', regex=True).apply(lambda x: x.str.strip())

df2.loc[df2['Артикул'] == '32925682', 'Оригинальный номер'] = '32925682'
df2.loc[df2['Артикул'] == '4312101', 'Артикул'] = '43121-01'
df2.loc[df2['Артикул'] == '438201', 'Артикул'] = '4382-01'
df2.loc[df2['Артикул'] == '438301', 'Артикул'] = '4383-01'
df2.loc[df2['Номенклатура'] == 'НПУ 00007940', 'Артикул'] = '85000159'
df2.loc[df2['Номенклатура'] == 'Насос подкачки 4128101', 'Артикул'] = '04128101'
df2.loc[df2['Номенклатура'] == 'Насос подкачки 4128101', 'Оригинальный номер'] = '04128101'
df2.loc[df2['Номенклатура'] == 'Фильтр масляный C-1049', 'Артикул'] = 'C1049'

mask_plus = (
    df2['Номенклатура'].str.contains(r'\+', na=False) &
    ~df2['Номенклатура'].str.contains(
        r'^(Колесо|РВД|[Оо]богреватель|Распределитель|Насос|Комплект|Кабель|Гидроцилиндр|Датчик|Коллектор|Фильтр топливный PERKINS|Фильтр воздушный \(внешний\+внутренний\) A5541S)',
        na=False
    )
)

mask_st_units = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111/40110',
    case=False,
    na=False
)
mask_brackets = df2['Номенклатура'].str.contains(
    r'(95*165*340/70*90*335,',
    case=False,
    na=False,
    regex=False
)

mask_kt = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный AF26656 к-т',
    case=False,
    na=False,
)

mask_AB = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40131AB',
    case=False,
    na=False,
)

mask_hid = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный ST40111ST40110',
    case=False,
    na=False,
)

mask_filter = df2['Номенклатура'].str.contains(
    r'Фильтр воздушный 6666375/6666376',
    case=False,
    na=False,
)

complects2 = df2[mask_plus | mask_st_units | mask_brackets | mask_kt | mask_AB | mask_hid | mask_filter]
complects2.loc[complects2['Номенклатура'] == 'Фильтр воздушный AF26656 к-т', 'Номенклатура'] = 'Фильтр воздушный к-т AF26656+AF26655'
complects2.loc[complects2['Номенклатура'] == 'Фильтр воздушный ST40131AB', 'Номенклатура'] = 'Фильтр воздушный (95*165*340/70*90*335, сдвоенный) kw1634'

df2 = df2[~df2.index.isin(complects2.index)]

In [312]:
complects2['Артикул'] = complects2['Номенклатура'].apply(lambda x: extract_articles(x, matches.keys()))

df_exploded = complects2.explode('Артикул').reset_index(drop=True)

df_exploded['Оригинальный номер'] = df_exploded['Артикул'].map(lambda x: matches[x].split()[0] if x in matches else None)

df_exploded[df_exploded['Артикул'].isna()]

assert (df_exploded.shape[0] == complects2.shape[0] * 2)

df2 = pd.concat([df2, df_exploded], ignore_index=True)

In [313]:
df2["Артикул"]  = df2["Артикул"].apply(normalize)
df2["Оригинальный номер"] = df2["Оригинальный номер"].apply(normalize)

for i in cols:
    df2[i] = df2[i].str.strip().str.upper()

def lookup(art_norm, orig_norm):

    if art_norm and art_norm in article_to_group:
        return article_to_group[art_norm], article_to_analogs[art_norm]
    if orig_norm and orig_norm in article_to_group:
        return article_to_group[orig_norm], article_to_analogs[orig_norm]
    return None, None

article_to_group   = {}
article_to_analogs = {}

for _, row in df1.drop_duplicates('Номенклатура.Артикул').iterrows():
    group = row['Номер группы']
    raw = row['all_analogs']
    try:
        analogs = raw if isinstance(raw, tuple) else ast.literal_eval(raw)
    except:
        analogs = ()
    for art in analogs:
        article_to_group[art]   = group
        article_to_analogs[art] = analogs
    main = row['Номенклатура.Артикул']
    if pd.notna(main) and main:
        article_to_group[main]   = group
        article_to_analogs[main] = analogs


groups, analogs_list = zip(
    *df2.apply(lambda r: lookup(r["Артикул"], r["Оригинальный номер"]), axis=1)
)

df2["Номер группы"] = groups
df2["Список аналогов"] = analogs_list

def enrich_analogs(row):
    art = row["Артикул"]
    analogs = row["Список аналогов"]
    if pd.isnull(art) or not isinstance(analogs, tuple):
        return analogs
    if art not in analogs and art not in article_to_group:
        return tuple(sorted(analogs + (art,)))
    return analogs

df2["Список аналогов"] = df2.apply(enrich_analogs, axis=1)
df2 = normalize_analog_lists(df2)

In [314]:
graph_new = defaultdict(set)

for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])
    if art in article_to_group or orig in article_to_group:
        continue

    if art is None:
        if orig is not None:
            art = orig
        else:
            continue 

    graph_new[art].add(art)

    if orig is not None and orig != art:
        graph_new[art].add(orig)
        graph_new[orig].add(art)


for idx in df2[df2["Номер группы"].isna()].index:
    art = normalize(df2.at[idx, "Артикул"])
    orig = normalize(df2.at[idx, "Оригинальный номер"])
    
    if art in article_to_group or (orig is not None and orig in article_to_group):
        continue
    if art is None:
        art = orig  

    if art is not None:
        df2.at[idx, "Список аналогов"] = find_all_analogs(art, graph_new)
    else:
        df2.at[idx, "Список аналогов"] = tuple()
        
unique_new_analogs = df2.loc[df2["Номер группы"].isna(), "Список аналогов"].drop_duplicates()
new_group = df1["Номер группы"].max() + 1
group_mapping = {grp: new_group + i for i, grp in enumerate(unique_new_analogs)}

mask = df2["Номер группы"].isna()
df2.loc[mask, "Номер группы"] = df2.loc[mask, "Список аналогов"].apply(
    lambda x: group_mapping.get(x)
)

In [315]:
def find_mixed_groups(df_raw: pd.DataFrame,
                      col_group: str = 'Номер группы',
                      col_name:  str = 'Номенклатура',
                      min_unique: int = 2) -> pd.DataFrame:
    """
    Найти группы аналогов, где одна группа содержит детали
    с принципиально разными названиями — признак ложного аналога.
    
    Применять на сырых данных ДО вызова merge_parts_df1/df2.
    """
    # Все уникальные названия по каждой группе
    group_names = (
        df_raw.groupby(col_group)[col_name]
        .apply(lambda s: sorted(s.dropna().astype(str).str.strip().unique()))
        .reset_index()
        .rename(columns={col_name: 'все_названия'})
    )
    group_names['кол_во_названий'] = group_names['все_названия'].apply(len)
    
    # Оставляем только группы с расхождением
    mixed = group_names[group_names['кол_во_названий'] >= min_unique].copy()
    
    # Считаем схожесть: если все названия похожи — это просто опечатки,
    # если нет — скорее всего разные детали
    import re
    def avg_similarity(names: list[str]) -> float:
        if len(names) < 2:
            return 1.0
        scores = []
        for i in range(len(names)):
            for j in range(i + 1, len(names)):
                w1 = set(re.split(r'\W+', names[i].lower())) - {''}
                w2 = set(re.split(r'\W+', names[j].lower())) - {''}
                scores.append(len(w1 & w2) / len(w1 | w2) if w1 | w2 else 1.0)
        return round(sum(scores) / len(scores), 2)
    
    mixed['схожесть_названий'] = mixed['все_названия'].apply(avg_similarity)
    
    return (
        mixed
        .sort_values('схожесть_названий')   # самые подозрительные первые
        [['Номер группы', 'кол_во_названий', 'схожесть_названий', 'все_названия']]
        .reset_index(drop=True)
    )

mixed = find_mixed_groups(df1)

suspicious = mixed[mixed['схожесть_названий'] <= 1]

result = (
    df1[df1['Номер группы'].isin(suspicious['Номер группы'])]
    [['Номер группы', 'Номенклатура.Артикул', 'Номенклатура']]
    .drop_duplicates()
    .sort_values('Номер группы')
)

# сохранить обычный файл
result.to_excel("mixed_groups.xlsx", index=False)

# ---- раскраска ----

group_ids = result['Номер группы'].ne(result['Номер группы'].shift()).cumsum()

palette = ["#FFCCCC","#CCFFCC","#CCCCFF","#FFFFCC","#CCFFFF"]

colors = {g: palette[i % len(palette)] for i, g in enumerate(group_ids.unique())}

def color_rows(row):
    color = colors[group_ids.loc[row.name]]
    return [f'background-color: {color}'] * len(row)

styled = result.style.apply(color_rows, axis=1)

styled.to_excel("colored.xlsx", engine="openpyxl", index=False)

PermissionError: [Errno 13] Permission denied: 'colored.xlsx'

In [ ]:
agg = merge_parts_df2(df2)
agg = prepare_sales(agg, df_quarter)

In [ ]:
final_df = pd.merge(
    df_quarter, 
    agg, 
    on=['Год','Месяц','Номер группы'], 
    how='outer', 
    suffixes=('_df','_agg'),
)

cols_to_combine = [c for c in df_quarter.columns if c in agg.columns and c not in ['Год','Месяц','Номер группы']]

for col in cols_to_combine:
    final_df[col] = final_df[f'{col}_df'].combine_first(final_df[f'{col}_agg'])

cols_to_drop = [c for c in final_df.columns if c.endswith('_df') or c.endswith('_agg')]

final_df.drop(columns=cols_to_drop, inplace=True)

final_df['Продажа'] = final_df['Продажа'].fillna(0)
final_df['Ремонт'] = final_df['Ремонт'].fillna(0)

cols = [c for c in final_df.columns if c not in ['Продажа', 'Ремонт', 'Приход', 'Конечный остаток']]

final_df = final_df[cols + ['Приход', 'Конечный остаток', 'Продажа', 'Ремонт']]

final_df = normalize_analog_lists(
    final_df,
    col_group="Номер группы",
    col_analogs="Список аналогов"
)
name_map = (
    final_df.groupby('Номер группы')['Номенклатура']
    .apply(lambda s: min(s.dropna().astype(str), key=len, default=None))
)
final_df['Номенклатура'] = final_df['Номер группы'].map(name_map)

In [ ]:
all_groups = []
seen = set()

for node in graph:
    if node not in seen:
        group_nodes = set(find_all_analogs(node, graph))  # преобразуем tuple в set
        all_groups.append(group_nodes)
        seen |= group_nodes

all_groups.sort(key=len, reverse=True)

top_n = 0
top_groups = all_groups[:top_n]

for i, group in enumerate(top_groups, 1):
    G = nx.Graph()
    for node in group:
        for neighbor in graph[node]:
            if neighbor in group:
                G.add_edge(node, neighbor)
    
    plt.figure(figsize=(16,10))
    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=True, node_size=2000, font_size=10)
    plt.title(f"Группа {i} — размер {len(group)}")
    plt.show()

In [ ]:
final_df.to_excel('final.xlsx',index=False)

- Внизу полная хуйня позорная

In [ ]:
recl = pd.read_excel('Аналитика по рекламации.xlsx', dtype=str)

mapping = str.maketrans({
    'А':'A','В':'B','Е':'E','К':'K','М':'M',
    'Н':'H','О':'O','Р':'P','С':'C','Т':'T','Х':'X'
})

# 1. Нормализуем df1
df1['M_norm'] = (
    df1['Машина']
    .astype(str)
    .str.upper()
    .str.translate(mapping)
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

# 2. Нормализуем recl
recl['Recl_norm'] = (
    recl['Модель']
    .astype(str)
    .str.upper()
    .str.translate(mapping)
    .str.replace(r"[^A-Z0-9]", "", regex=True)
)

# 3. Создаем список нормализованных машин + каталожный номер
recl_list = recl[['Recl_norm', 'Каталожный номер ЗЧ']].values.tolist()

# 4. Функция для поиска совпадений
def get_recl_cat_nums(m_norm):
    matches = [cat for recl_val, cat in recl_list if recl_val in m_norm]
    return ",".join(matches)  # объединяем через запятую

# 5. Применяем к df1
df1['recl_cat_nums'] = df1['M_norm'].apply(get_recl_cat_nums)

In [316]:
def _parse_analogs(val: Any) -> list[str]:
    """Распарсить поле «Список аналогов» в список строк."""
    if val is None or (isinstance(val, float) and np.isnan(val)):
        return []
    s = str(val).strip()
    if s in ("", "nan", "None"):
        return []
    try:
        parsed = ast.literal_eval(s)
        if isinstance(parsed, (tuple, list)):
            return [str(x).strip() for x in parsed if str(x).strip()]
        return [str(parsed).strip()]
    except (ValueError, SyntaxError):
        return [s]


def _norm(art: str) -> str:
    """Нормализация артикула для сравнений: верхний регистр, без пробелов."""
    return str(art).strip().upper()


def _name_similarity(n1: str, n2: str) -> float:
    """Доля общих слов относительно объединения множеств слов двух строк."""
    if not n1 or not n2:
        return 0.0
    w1 = set(re.split(r'\W+', n1.lower())) - {""}
    w2 = set(re.split(r'\W+', n2.lower())) - {""}
    if not w1 or not w2:
        return 0.0
    return len(w1 & w2) / len(w1 | w2)


def _confidence(sim: float) -> str:
    if sim >= 0.5:
        return "HIGH"
    if sim >= 0.15:
        return "MEDIUM"
    return "LOW"


# ---------------------------------------------------------------------------
# Главная функция аудита
# ---------------------------------------------------------------------------

def audit_analog_groups(
    df: pd.DataFrame,
    col_group: str = "Номер группы",
    col_article: str = "Артикул",
    col_name: str = "Номенклатура",
    col_analogs: str = "Список аналогов",
    # Суффиксы для LETTER_SUFFIX_SPLIT — проверяются как суффикс И как префикс
    letter_suffixes: tuple[str, ...] = (
        "A", "А",             # латиница + кириллица
        "B", "C", "F", "K",   # другие буквенные ревизии
        "GT", "IN",           # составные буквенные суффиксы
        " Б/У",               # бывшие в употреблении
        "-01", "-02", "-03",
        "-R", "-L", "-L1",
        " JR", " NSK",
    ),
    # Строки, которые проверяются ТОЛЬКО как префикс (не как суффикс)
    prefix_only: tuple[str, ...] = (
        "W",                  # WG0300002 vs G0300002
        "AVX",                # AVX10X1125 vs 10X1125
        "4Т-",                # 4Т-33006 vs 33006
    ),
    # Числовые суффиксы для NUMERIC_SUFFIX_SPLIT
    numeric_suffix_max_len: int = 3,  # 001, 01, 1 — но не 0001
    numeric_suffix_min_len: int = 1,
    # Максимальное кол-во ведущих нулей для LEADING_ZEROS_SPLIT
    leading_zeros_max: int = 4,
) -> pd.DataFrame:

    unique = (
        df[[col_group, col_article, col_name, col_analogs]]
        .drop_duplicates(subset=[col_group])
        .copy()
        .reset_index(drop=True)
    )

    unique["_parsed"] = unique[col_analogs].apply(_parse_analogs)
    unique["_name_lc"] = unique[col_name].fillna("").astype(str).str.strip().str.lower()

    # Вспомогательные индексы
    # article (norm) -> group
    art_to_group: dict[str, str] = {}
    # group -> set of norm articles in its analog list
    group_to_arts: dict[str, set[str]] = {}
    # group -> row (для быстрого доступа к имени)
    group_to_row: dict[str, pd.Series] = {}

    for _, row in unique.iterrows():
        grp = str(row[col_group])
        group_to_row[grp] = row
        arts = {_norm(a) for a in row["_parsed"]}
        group_to_arts[grp] = arts
        for a in arts:
            # Если коллизия — запишем специальный маркер
            if a in art_to_group and art_to_group[a] != grp:
                art_to_group[a] = "__DUPLICATE__"
            else:
                art_to_group[a] = grp

    issues: list[dict] = []

    def _row_info(grp: str) -> tuple[str, str, str]:
        """(article, name, analogs_str) для группы."""
        r = group_to_row[grp]
        return (
            str(r[col_article]),
            str(r[col_name]),
            str(r[col_analogs]),
        )

    def _add(
        issue_type: str,
        grp_a: str,
        grp_b: str | None,
        art_a: str,
        art_b: str | None,
        sim: float,
        comment: str,
        suffix_or_prefix: str = "",
    ):
        art_a_main, name_a, analogs_a = _row_info(grp_a)
        art_b_main, name_b, analogs_b = _row_info(grp_b) if grp_b else ("", "", "")
        issues.append(
            {
                "issue_type": issue_type,
                "confidence": _confidence(sim),
                "name_similarity": round(sim, 2),
                "group_a": grp_a,
                "group_b": grp_b or "",
                "article_a": art_a,
                "article_b": art_b or "",
                "suffix_or_prefix": suffix_or_prefix,
                "name_a": name_a,
                "name_b": name_b,
                "analogs_a": analogs_a,
                "analogs_b": analogs_b,
                "comment": comment,
            }
        )

    dup_arts = {a for a, g in art_to_group.items() if g == "__DUPLICATE__"}
    if dup_arts:
        # Нужно восстановить, в каких именно группах артикул встречается
        art_to_all_groups: dict[str, list[str]] = defaultdict(list)
        for _, row in unique.iterrows():
            grp = str(row[col_group])
            for a in row["_parsed"]:
                an = _norm(a)
                if an in dup_arts:
                    art_to_all_groups[an].append(grp)

        seen_dup_pairs: set[tuple] = set()
        for art, groups in art_to_all_groups.items():
            for i in range(len(groups)):
                for j in range(i + 1, len(groups)):
                    ga, gb = groups[i], groups[j]
                    pair = tuple(sorted([ga, gb]))
                    if pair in seen_dup_pairs:
                        continue
                    seen_dup_pairs.add(pair)
                    n_a = group_to_row[ga][col_name]
                    n_b = group_to_row[gb][col_name]
                    sim = _name_similarity(n_a, n_b)
                    _add(
                        "DUPLICATE_IN_GROUPS",
                        ga, gb, art, art, sim,
                        f"Артикул «{art}» встречается в аналогах обеих групп",
                    )

    for _, row in unique.iterrows():
        grp = str(row[col_group])
        own = _norm(str(row[col_article]))
        if own in ("NAN", "", "NONE"):
            continue
        if own not in group_to_arts[grp]:
            _add(
                "SELF_MISSING",
                grp, None, str(row[col_article]), None, 0.0,
                f"Собственный артикул «{row[col_article]}» отсутствует в списке аналогов группы",
            )


    all_arts_norm = list(art_to_group.keys())
    seen_letter_pairs: set[tuple] = set()

    # Суффиксная часть
    for base_art in all_arts_norm:
        if art_to_group.get(base_art) == "__DUPLICATE__":
            continue
        for suf in letter_suffixes:
            candidate = base_art + suf.upper()
            if candidate not in art_to_group:
                continue
            if art_to_group.get(candidate) == "__DUPLICATE__":
                continue
            ga = art_to_group[base_art]
            gb = art_to_group[candidate]
            if ga == gb:
                continue
            pair = tuple(sorted([ga + "|" + base_art, gb + "|" + candidate]))
            if pair in seen_letter_pairs:
                continue
            seen_letter_pairs.add(pair)
            n_a = group_to_row[ga][col_name]
            n_b = group_to_row[gb][col_name]
            sim = _name_similarity(n_a, n_b)
            _add(
                "LETTER_SUFFIX_SPLIT",
                ga, gb, base_art, candidate, sim,
                f"«{base_art}» и «{candidate}» (суффикс «{suf.strip()}») находятся в разных группах",
                suffix_or_prefix=suf.strip(),
            )

    all_prefixes = [
        s for s in letter_suffixes
        if not s.startswith(" ")
    ] + list(prefix_only)
    seen_prefix_pairs: set[tuple] = set()

    for base_art in all_arts_norm:
        if art_to_group.get(base_art) == "__DUPLICATE__":
            continue
        # base_art должен быть достаточно длинным, чтобы префикс имел смысл
        if len(base_art) < 3:
            continue
        for pre in all_prefixes:
            candidate = pre.upper() + base_art
            if candidate not in art_to_group:
                continue
            if art_to_group.get(candidate) == "__DUPLICATE__":
                continue
            ga = art_to_group[base_art]
            gb = art_to_group[candidate]
            if ga == gb:
                continue
            pair = tuple(sorted([ga + "|" + base_art, gb + "|" + candidate]))
            if pair in seen_prefix_pairs:
                continue
            seen_prefix_pairs.add(pair)
            n_a = group_to_row[ga][col_name]
            n_b = group_to_row[gb][col_name]
            sim = _name_similarity(n_a, n_b)
            _add(
                "LETTER_SUFFIX_SPLIT",
                ga, gb, base_art, candidate, sim,
                f"«{base_art}» и «{candidate}» (префикс «{pre.strip()}») находятся в разных группах",
                suffix_or_prefix=f"prefix:{pre.strip()}",
            )

    seen_num_pairs: set[tuple] = set()

    for base_art in all_arts_norm:
        if art_to_group.get(base_art) == "__DUPLICATE__":
            continue
        ga = art_to_group[base_art]
        for candidate in all_arts_norm:
            if candidate == base_art:
                continue
            if art_to_group.get(candidate) == "__DUPLICATE__":
                continue
            gb = art_to_group[candidate]
            if ga == gb:
                continue
            if not candidate.startswith(base_art):
                continue
            suffix = candidate[len(base_art):]
            if not (
                numeric_suffix_min_len <= len(suffix) <= numeric_suffix_max_len
                and suffix.isdigit()
            ):
                continue
            pair = tuple(sorted([ga + "|" + base_art, gb + "|" + candidate]))
            if pair in seen_num_pairs:
                continue
            seen_num_pairs.add(pair)
            n_a = group_to_row[ga][col_name]
            n_b = group_to_row[gb][col_name]
            sim = _name_similarity(n_a, n_b)
            _add(
                "NUMERIC_SUFFIX_SPLIT",
                ga, gb, base_art, candidate, sim,
                f"«{base_art}» и «{candidate}» (числовой суффикс «{suffix}») находятся в разных группах",
                suffix_or_prefix=suffix,
            )

    seen_zero_pairs: set[tuple] = set()

    for base_art in all_arts_norm:
        if art_to_group.get(base_art) == "__DUPLICATE__":
            continue
        # base_art не должен сам начинаться с нуля (иначе будет дублирование)
        if base_art.startswith("0"):
            continue
        ga = art_to_group[base_art]
        for n_zeros in range(1, leading_zeros_max + 1):
            candidate = "0" * n_zeros + base_art
            if candidate not in art_to_group:
                continue
            if art_to_group.get(candidate) == "__DUPLICATE__":
                continue
            gb = art_to_group[candidate]
            if ga == gb:
                continue
            pair = tuple(sorted([ga + "|" + base_art, gb + "|" + candidate]))
            if pair in seen_zero_pairs:
                continue
            seen_zero_pairs.add(pair)
            n_a = group_to_row[ga][col_name]
            n_b = group_to_row[gb][col_name]
            sim = _name_similarity(n_a, n_b)
            zeros = "0" * n_zeros
            _add(
                "LEADING_ZEROS_SPLIT",
                ga, gb, base_art, candidate, sim,
                f"«{base_art}» и «{candidate}» отличаются ведущими нулями «{zeros}»",
                suffix_or_prefix=f"prefix:{zeros}",
            )

    name_to_groups: dict[str, list[str]] = defaultdict(list)
    for _, row in unique.iterrows():
        name = row["_name_lc"]
        if name and name not in ("nan", "none", ""):
            name_to_groups[name].append(str(row[col_group]))

    seen_name_pairs: set[tuple] = set()
    for name, groups in name_to_groups.items():
        for i in range(len(groups)):
            for j in range(i + 1, len(groups)):
                ga, gb = groups[i], groups[j]
                pair = tuple(sorted([ga, gb]))
                if pair in seen_name_pairs:
                    continue
                seen_name_pairs.add(pair)
                art_a = str(group_to_row[ga][col_article])
                art_b = str(group_to_row[gb][col_article])
                _add(
                    "SAME_NAME_SPLIT",
                    ga, gb, art_a, art_b, 1.0,
                    f"Одинаковое название «{name}» в двух разных группах",
                )

    main_art_to_group: dict[str, str] = {}
    for _, row in unique.iterrows():
        an = _norm(str(row[col_article]))
        if an not in ("NAN", "", "NONE"):
            main_art_to_group[an] = str(row[col_group])

    seen_xref_pairs: set[tuple] = set()
    for _, row in unique.iterrows():
        ga = str(row[col_group])
        arts_a = group_to_arts[ga]
        for art in arts_a:
            if art not in main_art_to_group:
                continue
            gb = main_art_to_group[art]
            if gb == ga:
                continue
            arts_b = group_to_arts[gb]
            # Асимметрия: A ссылается на B через art, но B не ссылается на A
            arts_a_minus_self = arts_a - {_norm(str(row[col_article]))}
            if not arts_a_minus_self.intersection(arts_b - {art}):
                pair = tuple(sorted([ga, gb]))
                if pair in seen_xref_pairs:
                    continue
                seen_xref_pairs.add(pair)
                n_a = str(row[col_name])
                n_b = str(group_to_row[gb][col_name])
                sim = _name_similarity(n_a, n_b)
                art_b_main = str(group_to_row[gb][col_article])
                _add(
                    "CROSS_REF_ASYMMETRY",
                    ga, gb,
                    str(row[col_article]), art_b_main,
                    sim,
                    f"Группа {ga} содержит «{art}» (главный арт. группы {gb}), "
                    f"но группа {gb} не содержит артикулов группы {ga}",
                )

    if not issues:
        return pd.DataFrame(
            columns=[
                "issue_type", "confidence", "name_similarity",
                "group_a", "group_b", "article_a", "article_b",
                "suffix_or_prefix",
                "name_a", "name_b", "analogs_a", "analogs_b", "comment",
            ]
        )

    result = pd.DataFrame(issues)

    # Сортировка: сначала HIGH, потом по типу проблемы
    conf_order = {"HIGH": 0, "MEDIUM": 1, "LOW": 2}
    type_order = {
        "DUPLICATE_IN_GROUPS": 0,
        "SELF_MISSING": 1,
        "SAME_NAME_SPLIT": 2,
        "LETTER_SUFFIX_SPLIT": 3,
        "NUMERIC_SUFFIX_SPLIT": 4,
        "LEADING_ZEROS_SPLIT": 5,
        "CROSS_REF_ASYMMETRY": 6,
    }
    result["_c_ord"] = result["confidence"].map(conf_order)
    result["_t_ord"] = result["issue_type"].map(type_order).fillna(99)
    result = (
        result.sort_values(["_c_ord", "_t_ord", "name_similarity"],
                           ascending=[True, True, False])
        .drop(columns=["_c_ord", "_t_ord"])
        .reset_index(drop=True)
    )

    return result


def print_audit_summary(audit_df: pd.DataFrame) -> None:
    """Печать читаемого отчёта по результатам audit_analog_groups()."""
    if audit_df.empty:
        print("✅ Проблем не обнаружено.")
        return

    total = len(audit_df)
    print(f"{'='*70}")
    print(f"  АУДИТ АНАЛОГОВ  —  найдено проблем: {total}")
    print(f"{'='*70}")

    for issue_type, group in audit_df.groupby("issue_type", sort=False):
        counts = group["confidence"].value_counts()
        high   = counts.get("HIGH", 0)
        medium = counts.get("MEDIUM", 0)
        low    = counts.get("LOW", 0)
        print(f"\n▶  {issue_type}  ({len(group)} шт.)  "
              f"HIGH={high}  MEDIUM={medium}  LOW={low}")
        has_suf = "suffix_or_prefix" in group.columns
        print(f"   {'Гр.A':>6}  {'Гр.B':>6}  {'Артикул A':<22}  {'Артикул B':<22}  {'Suf/Pre':<10}  {'Conf':<7}  Sim")
        print(f"   {'-'*6}  {'-'*6}  {'-'*22}  {'-'*22}  {'-'*10}  {'-'*7}  ---")
        for _, r in group.iterrows():
            suf = str(r.get("suffix_or_prefix", ""))[:10] if has_suf else ""
            print(
                f"   {r['group_a']:>6}  {str(r['group_b']):>6}  "
                f"{str(r['article_a'])[:22]:<22}  {str(r['article_b'])[:22]:<22}  "
                f"{suf:<10}  {r['confidence']:<7}  {r['name_similarity']:.2f}"
            )
            print(f"         ↳ {r['name_a'][:60]}")
            if r["group_b"]:
                print(f"           {r['name_b'][:60]}")

    print(f"\n{'='*70}")
    print("Колонки результирующего датафрейма:")
    print("  issue_type, confidence, name_similarity,")
    print("  group_a, group_b, article_a, article_b, suffix_or_prefix,")
    print("  name_a, name_b, analogs_a, analogs_b, comment")
    print(f"{'='*70}")



def export_to_excel(audit_df: pd.DataFrame, path: str) -> None:
    try:
        from openpyxl import Workbook
        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
        from openpyxl.utils import get_column_letter
    except ImportError as e:
        raise ImportError("Установите openpyxl: pip install openpyxl") from e

    # ── Константы ──────────────────────────────────────────────────────────
    CLR = {
        "header_bg":    "1F3864",
        "header_fg":    "FFFFFF",
        "HIGH_bg":      "FFD7D7",
        "HIGH_fg":      "C00000",
        "MEDIUM_bg":    "FFF2CC",
        "MEDIUM_fg":    "7F6000",
        "LOW_bg":       "E2EFDA",
        "LOW_fg":       "375623",
        "alt_row":      "F2F7FC",
        "summary_hdr":  "2E75B6",
        "border":       "BFBFBF",
    }
    TAB_COLORS = {
        "SAME_NAME_SPLIT":      "C00000",
        "LETTER_SUFFIX_SPLIT":  "FF7C00",
        "NUMERIC_SUFFIX_SPLIT": "FFD700",
        "LEADING_ZEROS_SPLIT":  "70AD47",
        "DUPLICATE_IN_GROUPS":  "7030A0",
        "SELF_MISSING":         "00B0F0",
        "CROSS_REF_ASYMMETRY":  "808080",
    }
    ISSUE_NAMES = {
        "SAME_NAME_SPLIT":      "Одинаковое название, разные группы",
        "LETTER_SUFFIX_SPLIT":  "Буквенный суффикс / префикс",
        "NUMERIC_SUFFIX_SPLIT": "Числовой суффикс (ревизия)",
        "LEADING_ZEROS_SPLIT":  "Ведущие нули",
        "DUPLICATE_IN_GROUPS":  "Дубликат артикула в нескольких группах",
        "SELF_MISSING":         "Артикул отсутствует в своих аналогах",
        "CROSS_REF_ASYMMETRY":  "Асимметричная перекрёстная ссылка",
    }
    ISSUE_ORDER = [
        "SAME_NAME_SPLIT", "LETTER_SUFFIX_SPLIT", "NUMERIC_SUFFIX_SPLIT",
        "LEADING_ZEROS_SPLIT", "DUPLICATE_IN_GROUPS", "SELF_MISSING",
        "CROSS_REF_ASYMMETRY",
    ]
    CONF_RU = {"HIGH": "🔴 Высокая", "MEDIUM": "🟡 Средняя", "LOW": "🟢 Низкая"}
    COLS = [
        ("confidence",       "Уровень",       12),
        ("group_a",          "Группа A",       10),
        ("article_a",        "Артикул A",      22),
        ("name_a",           "Название A",     38),
        ("group_b",          "Группа B",       10),
        ("article_b",        "Артикул B",      22),
        ("name_b",           "Название B",     38),
        ("suffix_or_prefix", "Суффикс/Пре.",   13),
        ("name_similarity",  "Схожесть",       10),
        ("comment",          "Комментарий",    45),
    ]

    THIN = Side(style="thin",   color=CLR["border"])
    MED  = Side(style="medium", color="888888")

    def _fill(hex_color: str) -> PatternFill:
        return PatternFill("solid", fgColor=hex_color)

    def _border(*sides: str) -> Border:
        return Border(**{s: THIN for s in sides})

    def _hdr_font(size: int = 11) -> Font:
        return Font(name="Arial", bold=True, color=CLR["header_fg"], size=size)

    def _body_font(bold: bool = False, color: str = "000000", size: int = 10) -> Font:
        return Font(name="Arial", bold=bold, color=color, size=size)

    def _write_header_row(ws, titles: list[str], row: int = 1) -> None:
        for c, title in enumerate(titles, 1):
            cell = ws.cell(row=row, column=c, value=title)
            cell.font      = _hdr_font()
            cell.fill      = _fill(CLR["header_bg"])
            cell.alignment = Alignment(horizontal="center", vertical="center",
                                       wrap_text=True)
            cell.border    = Border(bottom=MED, top=MED, left=THIN, right=THIN)
        ws.row_dimensions[row].height = 30

    # ── Книга ──────────────────────────────────────────────────────────────
    wb = Workbook()
    wb.remove(wb.active)

    # ── Лист «Сводка» ──────────────────────────────────────────────────────
    ws_sum = wb.create_sheet("📊 Сводка")
    ws_sum.sheet_properties.tabColor = CLR["summary_hdr"]
    ws_sum.freeze_panes = "A3"

    ws_sum.merge_cells("A1:F1")
    tc = ws_sum["A1"]
    tc.value     = "АУДИТ АНАЛОГОВ — СВОДНЫЙ ОТЧЁТ"
    tc.font      = Font(name="Arial", bold=True, size=14, color="FFFFFF")
    tc.fill      = _fill(CLR["header_bg"])
    tc.alignment = Alignment(horizontal="center", vertical="center")
    ws_sum.row_dimensions[1].height = 36

    _write_header_row(
        ws_sum,
        ["Тип проблемы", "Описание", "🔴 Высокая", "🟡 Средняя", "🟢 Низкая", "Всего"],
        row=2,
    )

    total_h = total_m = total_l = 0
    summary_rows: list[list] = []
    for issue_type in ISSUE_ORDER:
        grp = audit_df[audit_df["issue_type"] == issue_type]
        if grp.empty:
            continue
        cnts = grp["confidence"].value_counts()
        h = int(cnts.get("HIGH",   0))
        m = int(cnts.get("MEDIUM", 0))
        l = int(cnts.get("LOW",    0))
        total_h += h; total_m += m; total_l += l
        summary_rows.append([issue_type, ISSUE_NAMES.get(issue_type, ""), h, m, l, h+m+l])

    for r_idx, row_data in enumerate(summary_rows, 3):
        alt = (r_idx % 2 == 0)
        for c_idx, val in enumerate(row_data, 1):
            cell = ws_sum.cell(row=r_idx, column=c_idx, value=val)
            cell.border    = _border("left", "right", "bottom")
            cell.alignment = Alignment(vertical="center",
                                       wrap_text=(c_idx <= 2))
            cell.fill = _fill(CLR["alt_row"]) if alt else PatternFill()
            if c_idx == 1:
                cell.font = _body_font(bold=True,
                                       color=TAB_COLORS.get(val, "000000"))
            elif c_idx == 3:
                cell.font = _body_font(bold=bool(val), color=CLR["HIGH_fg"])
            elif c_idx == 4:
                cell.font = _body_font(bold=bool(val), color=CLR["MEDIUM_fg"])
            elif c_idx == 5:
                cell.font = _body_font(bold=bool(val), color=CLR["LOW_fg"])
            elif c_idx == 6:
                cell.font = _body_font(bold=True)
            else:
                cell.font = _body_font()

    # итоговая строка
    tot = len(summary_rows) + 3
    for c_idx, val in enumerate(
        ["ИТОГО", "", total_h, total_m, total_l, total_h + total_m + total_l], 1
    ):
        cell = ws_sum.cell(row=tot, column=c_idx, value=val)
        cell.font   = _body_font(bold=True, size=11)
        cell.fill   = _fill("D9E1F2")
        cell.border = Border(top=MED, bottom=MED, left=THIN, right=THIN)
        cell.alignment = Alignment(horizontal="center", vertical="center")

    ws_sum.column_dimensions["A"].width = 26
    ws_sum.column_dimensions["B"].width = 42
    for col in ["C", "D", "E", "F"]:
        ws_sum.column_dimensions[col].width = 14

    # ── Листы по типам ─────────────────────────────────────────────────────
    for issue_type in ISSUE_ORDER:
        grp = audit_df[audit_df["issue_type"] == issue_type]
        if grp.empty:
            continue

        tab_name = issue_type.replace("_SPLIT", "").replace("_", " ")[:28]
        ws = wb.create_sheet(tab_name)
        ws.sheet_properties.tabColor = TAB_COLORS.get(issue_type, "808080")
        ws.freeze_panes = "A3"

        # строка-заголовок листа
        n_cols = len(COLS)
        ws.merge_cells(start_row=1, start_column=1,
                       end_row=1,   end_column=n_cols)
        tc = ws.cell(row=1, column=1,
                     value=f"{ISSUE_NAMES.get(issue_type, issue_type)}  ({len(grp)} шт.)")
        tc.font      = Font(name="Arial", bold=True, size=12, color="FFFFFF")
        tc.fill      = _fill(TAB_COLORS.get(issue_type, "808080"))
        tc.alignment = Alignment(horizontal="left", vertical="center", indent=1)
        ws.row_dimensions[1].height = 28

        _write_header_row(ws, [c[1] for c in COLS], row=2)

        CENTER_FIELDS = {"confidence", "group_a", "group_b",
                         "suffix_or_prefix", "name_similarity"}
        WRAP_FIELDS   = {"name_a", "name_b", "comment"}

        for r_idx, (_, row_data) in enumerate(grp.iterrows(), 3):
            conf = str(row_data.get("confidence", "LOW"))
            alt  = (r_idx % 2 == 0)

            for c_idx, (field, _, _width) in enumerate(COLS, 1):
                val = row_data.get(field, "")
                if val is None or (isinstance(val, float) and np.isnan(val)):
                    val = ""
                if field == "confidence":
                    val = CONF_RU.get(str(val), val)
                if field == "name_similarity" and val != "":
                    try:
                        val = float(val)
                    except (TypeError, ValueError):
                        pass

                cell = ws.cell(row=r_idx, column=c_idx, value=val)
                cell.border    = _border("left", "right", "bottom")
                cell.alignment = Alignment(
                    vertical="center",
                    wrap_text=(field in WRAP_FIELDS),
                    horizontal="center" if field in CENTER_FIELDS else "left",
                )

                if field == "confidence":
                    cell.fill = _fill(CLR[f"{conf}_bg"])
                    cell.font = _body_font(bold=True, color=CLR[f"{conf}_fg"])
                elif alt:
                    cell.fill = _fill(CLR["alt_row"])
                    cell.font = _body_font()
                else:
                    cell.font = _body_font()

                if field == "name_similarity" and val != "":
                    cell.number_format = "0.00"

            ws.row_dimensions[r_idx].height = 28

        for c_idx, (_, _, w) in enumerate(COLS, 1):
            ws.column_dimensions[get_column_letter(c_idx)].width = w

    wb.save(path)
    print(f"✅ Отчёт сохранён: {path}  ({len(audit_df)} проблем, {len(wb.sheetnames)} листов)")

df = pd.read_excel("final.xlsx", dtype=str)
result = audit_analog_groups(df)
export_to_excel(result, "audit_report.xlsx")

✅ Отчёт сохранён: audit_report.xlsx  (87 проблем, 5 листов)


In [ ]:
"""
=============================================================
  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026
  С автоматической обработкой выбросов (IQR-метод)
=============================================================
Использование:
  python forecast_vs_fact.py          # топ-10 по умолчанию
  python forecast_vs_fact.py 9 146    # конкретные группы

Требования:
  pip install pandas openpyxl statsmodels

Файл данных: final.xlsx — в той же папке, что и скрипт.
=============================================================
"""

import pandas as pd
import numpy as np
import ast
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import sys

warnings.filterwarnings('ignore')

DATA_FILE = "final.xlsx"

# Прогноз на Дек 2025, Янв 2026, Фев 2026
FORECAST_MONTHS = [(2025, 12), (2026, 1), (2026, 2)]
FORECAST_LABELS = ["Дек 2025", "Янв 2026", "Фев 2026"]

# Обучение: всё до декабря 2025
TRAIN_END_YEAR, TRAIN_END_MONTH = 2025, 11


# ── Загрузка ──────────────────────────────────────────────────────────────────
def load_data(filepath):
    df = pd.read_excel(filepath)
    df_train = df[
        ~((df['Год'] == 2025) & (df['Месяц'] == 12)) &
        ~(df['Год'] == 2026)
    ].copy()
    df_train['date'] = pd.to_datetime(
        df_train['Год'].astype(str) + '-' +
        df_train['Месяц'].astype(str).str.zfill(2) + '-01'
    )
    monthly = (df_train
               .groupby(['date', 'Номер группы'])[['Продажа', 'Ремонт']]
               .sum()
               .reset_index())

    # Факт за Дек 25 / Янв 26 / Фев 26
    actual = {}
    for y, m in FORECAST_MONTHS:
        sub = df[(df['Год'] == y) & (df['Месяц'] == m)]
        for _, row in sub.iterrows():
            g = int(row['Номер группы']) if not pd.isna(row['Номер группы']) else None
            if g is None:
                continue
            if g not in actual:
                actual[g] = {'sale': [0, 0, 0], 'repair': [0, 0, 0]}
            idx = FORECAST_MONTHS.index((y, m))
            actual[g]['sale'][idx]   += float(row['Продажа'] or 0)
            actual[g]['repair'][idx] += float(row['Ремонт']  or 0)

    # Мета
    def parse_analogs(v):
        if pd.isna(v) or str(v).strip() in ('', 'nan'):
            return ''
        try:
            return ', '.join(str(x) for x in ast.literal_eval(str(v)))
        except Exception:
            return str(v).strip("()'\"")

    group_meta = (df.groupby('Номер группы')
                  .agg(Номенклатура=('Номенклатура', 'first'),
                       Артикул=('Артикул', 'first'),
                       ТипТехники=('Тип техники', 'first'),
                       all_analogs=('all_analogs', 'first'))
                  .reset_index())
    group_meta['Аналоги'] = group_meta['all_analogs'].apply(parse_analogs)
    group_meta['Номер группы'] = group_meta['Номер группы'].astype(int)

    all_groups = set(group_meta['Номер группы'].tolist())
    return df_train, monthly, group_meta, actual, all_groups


# ── Обнаружение и замена выбросов ─────────────────────────────────────────────
def remove_outliers(series: pd.Series, iqr_factor: float = 1.5) -> tuple:
    """
    Заменяет выбросы (> Q3 + factor*IQR) на скользящую медиану по 3 соседям.
    Возвращает (очищенный ряд, список выбросов [(дата, исходное, замена)]).
    """
    s = series.copy()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + iqr_factor * iqr

    # Нижний порог не используем (нули — норма для запчастей)
    outlier_mask = s > upper

    outlier_log = []
    for idx in s[outlier_mask].index:
        original = s[idx]
        # Медиана ближайших 3 соседей (без самого выброса)
        pos = s.index.get_loc(idx)
        neighbors = []
        for offset in [-3, -2, -1, 1, 2, 3]:
            ni = pos + offset
            if 0 <= ni < len(s) and not outlier_mask.iloc[ni]:
                neighbors.append(s.iloc[ni])
        replacement = float(np.median(neighbors)) if neighbors else float(s.median())
        s.iloc[pos] = replacement
        outlier_log.append({
            'date':        idx.strftime('%Y-%m'),
            'original':    round(original, 1),
            'replacement': round(replacement, 1),
            'threshold':   round(upper, 1),
        })

    return s, outlier_log


# ── Прогнозирование ───────────────────────────────────────────────────────────
def get_series(monthly, grp, col):
    full_dates = pd.date_range('2023-01-01', '2025-11-01', freq='MS')
    sub = (monthly[monthly['Номер группы'] == grp][['date', col]]
           .set_index('date'))
    return sub[col].reindex(full_dates, fill_value=0).astype(float)


def forecast_hw(series_clean, steps=3):
    s = series_clean.clip(lower=0)
    best_fc, best_aic, best_name = None, np.inf, 'Mean-6m'
    for trend, seasonal, name in [
        ('add', 'add', 'HW-add'),
        ('add', 'mul', 'HW-mul'),
    ]:
        if seasonal == 'mul' and s.min() <= 0:
            continue
        try:
            m = ExponentialSmoothing(s, trend=trend, seasonal=seasonal,
                                     seasonal_periods=12)
            f = m.fit(optimized=True)
            if f.aic < best_aic:
                best_aic  = f.aic
                best_fc   = f.forecast(steps).clip(lower=0).round(1)
                best_name = name
        except Exception:
            pass
    if best_fc is None:
        v = s.tail(6).mean()
        best_fc = pd.Series([v] * steps,
                            index=pd.date_range('2025-12-01',
                                                periods=steps, freq='MS')).round(1)
    return best_fc, best_name


def forecast_group(monthly, group_meta, actual, grp):
    meta_row = group_meta[group_meta['Номер группы'] == grp]
    if meta_row.empty:
        return None
    meta = meta_row.iloc[0]

    result = {'grp': grp,
              'name':    str(meta['Номенклатура'])[:70],
              'article': str(meta['Артикул']),
              'type':    str(meta['ТипТехники']),
              'analogs': str(meta['Аналоги']),
              'sale':   {},
              'repair': {}}

    for col, key in [('Продажа', 'sale'), ('Ремонт', 'repair')]:
        raw = get_series(monthly, grp, col)

        # — Без обработки выбросов
        fc_raw, method_raw = forecast_hw(raw, 3)

        # — С обработкой выбросов
        cleaned, outlier_log = remove_outliers(raw)
        fc_clean, method_clean = forecast_hw(cleaned, 3)

        act_vals = actual.get(grp, {}).get(key, [0, 0, 0])

        result[key] = {
            'fc_raw':      [round(float(fc_raw.iloc[i]),   1) for i in range(3)],
            'fc_clean':    [round(float(fc_clean.iloc[i]), 1) for i in range(3)],
            'act':         [round(float(v), 1) for v in act_vals],
            'method_raw':  method_raw,
            'method_clean': method_clean,
            'outliers':    outlier_log,
        }
    return result


def build_forecast_rows(groups, monthly, group_meta, actual, col_key):
    rows = []
    for grp in groups:
        print(f"    → Группа {grp}...", end='', flush=True)
        r = forecast_group(monthly, group_meta, actual, grp)
        if r:
            rows.append(r)
            d = r[col_key]
            n_out = len(d['outliers'])
            print(f"  {n_out} выбросов | "
                  f"fc_raw={d['fc_raw']} | fc_clean={d['fc_clean']} | act={d['act']}")
        else:
            print(" — нет данных")
    return rows


# ── Excel ─────────────────────────────────────────────────────────────────────
C_DARK = "1A2744"; C_MID = "2D4A8A"; C_ACC = "4A90D9"
C_ORG  = "B8520A"; C_ORL = "FFF3CD"
C_GRN  = "1E6E42"; C_GNL = "E9F7EF"
C_RED  = "922B21"; C_RDL = "FDEDEC"
C_STR  = "EBF3FB"; C_WHT = "FFFFFF"
C_TL   = "117A65"; C_TLL = "D5F5E3"
C_PRP  = "5B2C6F"; C_PRL = "F4ECF7"  # purple for cleaned forecast

def fl(c):  return PatternFill("solid", fgColor=c)
def fn(bold=False, color="000000", size=9, italic=False):
    return Font(name='Arial', bold=bold, color=color, size=size, italic=italic)
_t  = Side(border_style="thin",   color="CCCCCC")
_tb = Side(border_style="medium", color="2D4A8A")
BRD = Border(left=_t, right=_t, top=_t, bottom=_t)
CTR = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT = Alignment(horizontal='left',   vertical='center', wrap_text=True)
RGT = Alignment(horizontal='right',  vertical='center')

def sc(ws, r, c, v, _fn=None, _fl=None, _al=None, fmt=None):
    cell = ws.cell(row=r, column=c, value=v)
    if _fn: cell.font = _fn
    if _fl: cell.fill = _fl
    if _al: cell.alignment = _al
    if fmt: cell.number_format = fmt
    cell.border = BRD

def mc(ws, r1, c1, r2, c2, v, _fn=None, _fl=None, _al=None):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    cell = ws.cell(row=r1, column=c1, value=v)
    if _fn: cell.font = _fn
    if _fl:
        for row in ws.iter_rows(min_row=r1, max_row=r2, min_col=c1, max_col=c2):
            for c in row: c.fill = _fl; c.border = BRD
    if _al: cell.alignment = _al


# ─────────────────────────────────────────────────────────────────────────────
# COL LAYOUT (per sheet):
#  1=№  2=Гр  3=Артикул  4=Номенклатура  5=Тип  6=Аналоги
#  Per month (×3):
#    col_base + 0 = Прогноз (raw)
#    col_base + 1 = Прогноз (без выбросов)
#    col_base + 2 = Факт
#    col_base + 3 = Δ raw
#    col_base + 4 = Δ clean
#  MONTH 1 cols: 7-11  MONTH 2: 12-16  MONTH 3: 17-21
#  ИТОГО: 22=fc_raw  23=fc_clean  24=Факт  25=Δraw  26=Δclean
# ─────────────────────────────────────────────────────────────────────────────

def build_sheet(ws, rows, col_label, col_key):
    ws.sheet_view.showGridLines = False
    NCOLS = 26

    # Title
    ws.row_dimensions[1].height = 28
    mc(ws, 1, 1, 1, NCOLS,
       f"ПРОГНОЗ vs ФАКТ — {col_label} | Дек 2025 – Фев 2026",
       _fn=fn(True, C_WHT, 13), _fl=fl(C_DARK), _al=CTR)

    ws.row_dimensions[2].height = 14
    mc(ws, 2, 1, 2, NCOLS,
       "Обучение: янв 2023 – ноя 2025 | "
       "Выбросы: IQR × 1.5 → заменяются медианой соседей | "
       "Δ = Факт – Прогноз | Зелёный = модель недооценила, Красный = переоценила",
       _fn=fn(italic=True, color="555555", size=8), _fl=fl("F0F4FA"), _al=CTR)

    # Header rows 4-5
    ws.row_dimensions[4].height = 20
    ws.row_dimensions[5].height = 18

    for c, v in [(1,"№"),(2,"Группа"),(3,"Артикул"),(4,"Номенклатура"),(5,"Тип"),(6,"Аналоги")]:
        mc(ws, 4, c, 5, c, v, _fn=fn(True, C_WHT, 9), _fl=fl(C_DARK), _al=CTR)

    month_colors = ["2C3E50", "1A252F", "0D1B2A"]
    for i, (lbl, bg) in enumerate(zip(FORECAST_LABELS, month_colors)):
        base = 7 + i * 5
        mc(ws, 4, base, 4, base+4, lbl,
           _fn=fn(True, C_WHT, 10), _fl=fl(bg), _al=CTR)
        sub_hdrs = [
            ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
            ("Факт",    C_TL),  ("Δ (осн)",  C_RED),
            ("Δ (чист)",C_GRN),
        ]
        for j, (sh, sc_c) in enumerate(sub_hdrs):
            sc(ws, 5, base+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    mc(ws, 4, 22, 4, 26, "ИТОГО КВАРТАЛ",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_GRN), _al=CTR)
    for j, (sh, sc_c) in enumerate([
        ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
        ("Факт", C_TL), ("Δ (осн)", C_RED), ("Δ (чист)", C_GRN)
    ]):
        sc(ws, 5, 22+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    # Data rows
    for idx, row in enumerate(rows):
        r = 6 + idx
        ws.row_dimensions[r].height = 28
        bg = C_STR if idx % 2 == 0 else C_WHT
        d  = row[col_key]

        sc(ws, r, 1, idx+1,          _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 2, row['grp'],      _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 3, row['article'],  _fn=fn(size=8),  _fl=fl(bg), _al=CTR)
        sc(ws, r, 4, row['name'],     _fn=fn(size=8),  _fl=fl(bg), _al=LFT)
        sc(ws, r, 5, row['type'],     _fn=fn(size=8),  _fl=fl(bg), _al=CTR)

        # Аналоги + выбросы
        out_note = ""
        if d['outliers']:
            parts = [f"{o['date']}: {o['original']:.0f}→{o['replacement']:.0f}"
                     for o in d['outliers']]
            out_note = " | ⚠ выбросы: " + "; ".join(parts)
        sc(ws, r, 6, row['analogs'] + out_note,
           _fn=fn(size=7), _fl=fl(bg), _al=LFT)

        sum_raw = 0; sum_clean = 0; sum_act = 0
        for i in range(3):
            base    = 7 + i * 5
            fc_raw  = round(d['fc_raw'][i],   1)
            fc_cln  = round(d['fc_clean'][i], 1)
            act_v   = round(d['act'][i],      1)
            d_raw   = round(act_v - fc_raw,   1)
            d_cln   = round(act_v - fc_cln,   1)
            sum_raw   += fc_raw
            sum_clean += fc_cln
            sum_act   += act_v

            sc(ws, r, base,   fc_raw, _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+1, fc_cln, _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+2, act_v,  _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')

            for delta, col_i in [(d_raw, base+3), (d_cln, base+4)]:
                clr = C_TL  if delta >= 0 else C_RED
                bg2 = C_TLL if delta >= 0 else C_RDL
                sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
                   _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

        # Totals
        td_raw  = round(sum_act - sum_raw,   1)
        td_cln  = round(sum_act - sum_clean, 1)
        sc(ws, r, 22, round(sum_raw,   1), _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 23, round(sum_clean, 1), _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 24, round(sum_act,   1), _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(td_raw, 25), (td_cln, 26)]:
            clr = C_TL  if delta >= 0 else C_RED
            bg2 = C_TLL if delta >= 0 else C_RDL
            sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

    # Grand total row
    tr = 6 + len(rows)
    ws.row_dimensions[tr].height = 20
    mc(ws, tr, 1, tr, 6, "ИТОГО ТОП-10",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_DARK), _al=CTR)

    for i in range(3):
        base = 7 + i * 5
        sf = round(sum(row[col_key]['fc_raw'][i]   for row in rows), 1)
        sc2= round(sum(row[col_key]['fc_clean'][i] for row in rows), 1)
        sa = round(sum(row[col_key]['act'][i]      for row in rows), 1)
        sc(ws, tr, base,   sf,  _fn=fn(True,C_WHT), _fl=fl(C_ORG), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+1, sc2, _fn=fn(True,C_WHT), _fl=fl(C_PRP), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+2, sa,  _fn=fn(True,C_WHT), _fl=fl(C_TL),  _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(round(sa-sf,1), base+3), (round(sa-sc2,1), base+4)]:
            sc(ws, tr, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True,C_WHT), _fl=fl(C_RED if delta<0 else C_TL), _al=CTR)

    tf  = round(sum(sum(row[col_key]['fc_raw'])   for row in rows), 1)
    tc  = round(sum(sum(row[col_key]['fc_clean']) for row in rows), 1)
    ta  = round(sum(sum(row[col_key]['act'])      for row in rows), 1)
    sc(ws,tr,22,tf,_fn=fn(True,C_WHT,10),_fl=fl(C_ORG),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,23,tc,_fn=fn(True,C_WHT,10),_fl=fl(C_PRP),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,24,ta,_fn=fn(True,C_WHT,10),_fl=fl(C_TL), _al=RGT,fmt='#,##0.0')
    for delta, col_i in [(round(ta-tf,1),25),(round(ta-tc,1),26)]:
        sc(ws,tr,col_i,f"{'+' if delta>=0 else ''}{delta:.1f}",
           _fn=fn(True,C_WHT,10),_fl=fl(C_RED if delta<0 else C_TL),_al=CTR)

    # MAPE row
    mr = tr + 1
    ws.row_dimensions[mr].height = 16
    mape_raw  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_raw'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mape_cln  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_clean'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mc(ws, mr, 1, mr, NCOLS,
       f"MAPE (средняя абсолютная % ошибка): "
       f"Прогноз (осн.) = {mape_raw:.1f}% | "
       f"Прогноз (без выбросов) = {mape_cln:.1f}%  "
       f"{'Очистка помогла' if mape_cln < mape_raw else '⚠ Очистка не улучшила'}",
       _fn=fn(bold=True, color=C_GRN if mape_cln < mape_raw else C_RED, size=9),
       _fl=fl("FDFEFE"), _al=LFT)

    # Column widths
    ws.column_dimensions['A'].width = 4
    ws.column_dimensions['B'].width = 7
    ws.column_dimensions['C'].width = 13
    ws.column_dimensions['D'].width = 34
    ws.column_dimensions['E'].width = 13
    ws.column_dimensions['F'].width = 40
    for i in range(3):
        for j, w in enumerate([10, 11, 10, 8, 8]):
            ws.column_dimensions[get_column_letter(7 + i*5 + j)].width = w
    for j, w in enumerate([10, 11, 10, 8, 8]):
        ws.column_dimensions[get_column_letter(22 + j)].width = w
    ws.freeze_panes = 'G6'


def build_outlier_sheet(ws, rows_sale, rows_rep):
    """Отдельный лист — подробная таблица всех выбросов."""
    ws.title = "⚠ Выбросы"
    ws.sheet_view.showGridLines = False

    ws.row_dimensions[1].height = 26
    mc(ws, 1, 1, 1, 8,
       "ОБНАРУЖЕННЫЕ ВЫБРОСЫ — автоматическая корректировка (IQR × 1.5)",
       _fn=fn(True, C_WHT, 12), _fl=fl(C_DARK), _al=CTR)

    hdrs = ["Группа", "Номенклатура", "Тип (прод/рем)",
            "Дата", "Исходное значение", "Порог IQR",
            "Заменено на", "Δ корректировка"]
    ws.row_dimensions[3].height = 18
    for ci, h in enumerate(hdrs):
        sc(ws, 3, 1+ci, h, _fn=fn(True, C_WHT, 9), _fl=fl(C_MID), _al=CTR)

    r = 4
    for rows, kind in [(rows_sale, "Продажи"), (rows_rep, "Ремонт")]:
        for row in rows:
            for o in row['sale' if kind == "Продажи" else 'repair']['outliers']:
                ws.row_dimensions[r].height = 16
                delta = round(o['replacement'] - o['original'], 1)
                bg = C_RDL
                sc(ws,r,1,row['grp'],          _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,2,row['name'][:50],    _fn=fn(size=8),_fl=fl(bg),_al=LFT)
                sc(ws,r,3,kind,                _fn=fn(size=8),_fl=fl(bg),_al=CTR)
                sc(ws,r,4,o['date'],           _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,5,o['original'],       _fn=fn(True,C_RED),_fl=fl(C_RDL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,6,o['threshold'],      _fn=fn(),      _fl=fl(bg),_al=RGT,fmt='#,##0.0')
                sc(ws,r,7,o['replacement'],    _fn=fn(True,C_GRN),_fl=fl(C_GNL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,8,f"{delta:+.1f}",     _fn=fn(True,C_RED),_fl=fl(bg),_al=CTR)
                r += 1

    for ci, w in enumerate([7,40,12,10,14,12,13,14]):
        ws.column_dimensions[get_column_letter(1+ci)].width = w
    ws.freeze_panes = 'A4'


# ── Точность прогноза (текстовый вывод) ──────────────────────────────────────
def print_accuracy(rows, col_key, label):
    print(f"\n  {'─'*65}")
    print(f"  ТОЧНОСТЬ — {label}")
    print(f"  {'Гр.':>5}  {'fc_raw Q3':>10}  {'fc_cln Q3':>10}  "
          f"{'Факт Q3':>9}  {'Δ raw':>7}  {'Δ clean':>8}  MAPE_raw  MAPE_cln")
    print(f"  {'─'*65}")
    for row in rows:
        d = row[col_key]
        fr = round(sum(d['fc_raw']),   1)
        fc = round(sum(d['fc_clean']), 1)
        fa = round(sum(d['act']),      1)
        dr = round(fa - fr, 1)
        dc = round(fa - fc, 1)
        mape_r = abs(dr) / (fa + 1e-9) * 100
        mape_c = abs(dc) / (fa + 1e-9) * 100
        better = "✅" if mape_c < mape_r else "  "
        print(f"  {row['grp']:>5}  {fr:>10.1f}  {fc:>10.1f}  "
              f"{fa:>9.1f}  {dr:>+7.1f}  {dc:>+8.1f}  "
              f"{mape_r:>7.1f}%  {mape_c:>7.1f}% {better}")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()

    data_path = os.path.join(script_dir, DATA_FILE)
    if not os.path.exists(data_path):
        print(f"\n❌  Файл не найден: {data_path}")
        sys.exit(1)

    print("\n" + "="*65)
    print("  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов")
    print("="*65)
    print(f"\n  Загрузка {DATA_FILE}...")
    df_train, monthly, group_meta, actual, all_groups = load_data(data_path)
    print(f"  Загружено. Групп: {len(all_groups)}")

    # ── Топ-20 для справки ────────────────────────────────────────────────────
    summary = (df_train.groupby('Номер группы')
               .agg(Продажа=('Продажа', 'sum'),
                    Ремонт=('Ремонт', 'sum'),
                    Номенклатура=('Номенклатура', 'first'))
               .reset_index()
               .sort_values('Продажа', ascending=False))

    print("\n  Топ-20 по продажам (для справки):")
    for _, row in summary.head(20).iterrows():
        print(f"    Гр.{int(row['Номер группы']):4d} | "
              f"Прод: {row['Продажа']:6.0f} | "
              f"Рем: {row['Ремонт']:5.0f} | "
              f"{str(row['Номенклатура'])[:50]}")

    def parse_input(raw):
        nums = set()
        raw = raw.replace(',', ' ')
        for part in raw.split():
            if '-' in part and not part.startswith('-'):
                a, b = part.split('-', 1)
                try: nums.update(range(int(a), int(b) + 1))
                except ValueError: pass
            else:
                try: nums.add(int(part))
                except ValueError: pass
        return sorted(nums)

    # ── Интерактивный цикл ────────────────────────────────────────────────────
    while True:
        print("\n" + "-"*65)
        print("  Форматы: '9 146 205'  |  '9,146'  |  '9-15'  |  'топ10'")
        print("  'поиск <слово>' — поиск по названию  |  'выход' — выйти")
        raw = input("\n  Номера групп: ").strip()

        if raw.lower() in ('выход', 'exit', 'q'):
            print("  До свидания!"); return

        if raw.lower().startswith('поиск'):
            q = raw[5:].strip() or input("  Слово для поиска: ").strip()
            found = summary[summary['Номенклатура'].str.contains(q, case=False, na=False)]
            if found.empty:
                print(f"  Ничего не найдено по: '{q}'")
            else:
                for _, r in found.iterrows():
                    print(f"    Гр.{int(r['Номер группы']):4d} | "
                          f"{r['Продажа']:6.0f} прод. | "
                          f"{str(r['Номенклатура'])[:55]}")
            continue

        if raw.lower() in ('топ10', 'топ', 'top'):
            requested = summary.head(10)['Номер группы'].astype(int).tolist()
        else:
            requested = parse_input(raw)

        if not requested:
            print("  ⚠️  Не удалось распознать номера. Попробуйте ещё раз.")
            continue

        valid   = [g for g in requested if g in all_groups]
        invalid = [g for g in requested if g not in all_groups]
        if invalid:
            print(f"  Не найдены в данных: {invalid}")
        if not valid:
            print("  Ни одна группа не найдена.")
            continue

        top10_sale = valid
        top10_rep  = valid
        print(f"\n  Выбранные группы: {valid}")

        print("\n  Строим прогноз...")
        all_results = {}
        for grp in list(dict.fromkeys(top10_sale + top10_rep)):
            print(f"    → Группа {grp}...", end='', flush=True)
            r = forecast_group(monthly, group_meta, actual, grp)
            if r:
                all_results[grp] = r
                d = r['sale']
                print(f"  {len(d['outliers'])} выбросов | "
                      f"fc={d['fc_raw']} → clean={d['fc_clean']} | fact={d['act']}")
            else:
                print("  нет данных")

        rows_sale = [all_results[g] for g in top10_sale if g in all_results]
        rows_rep  = [all_results[g] for g in top10_rep  if g in all_results]

        print_accuracy(rows_sale, 'sale',   'ПРОДАЖИ')
        print_accuracy(rows_rep,  'repair', 'РЕМОНТ')

        wb = openpyxl.Workbook()
        ws1 = wb.active; ws1.title = "Продажи"
        build_sheet(ws1, rows_sale, "ПРОДАЖИ", "sale")
        ws2 = wb.create_sheet("🔧 Ремонт")
        build_sheet(ws2, rows_rep, "РЕМОНТ", "repair")
        ws3 = wb.create_sheet()
        build_outlier_sheet(ws3, rows_sale, rows_rep)

        try:
            _sdir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            _sdir = os.getcwd()
        groups_str = '_'.join(str(g) for g in valid[:5])
        out_path = os.path.join(_sdir, f"forecast_vs_fact_{groups_str}.xlsx")
        wb.save(out_path)
        print(f"\n  Файл сохранён: {out_path}")

        again = input("\n  Ещё один прогноз? (да/нет): ").strip().lower()
        if again not in ('да', 'д', 'y', 'yes'):
            return


if __name__ == "__main__":
    main()


  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов

  Загрузка final.xlsx...


KeyError: "Column(s) ['all_analogs'] do not exist"

In [ ]:
"""
=============================================================
  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026
  С автоматической обработкой выбросов (IQR-метод)
=============================================================
Использование:
  python forecast_vs_fact.py          # топ-10 по умолчанию
  python forecast_vs_fact.py 9 146    # конкретные группы

Требования:
  pip install pandas openpyxl statsmodels

Файл данных: final.xlsx — в той же папке, что и скрипт.
=============================================================
"""

import pandas as pd
import numpy as np
import ast
import openpyxl
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter
from statsmodels.tsa.holtwinters import ExponentialSmoothing
import warnings
import os
import sys

warnings.filterwarnings('ignore')

DATA_FILE = "final.xlsx"

# Прогноз на Дек 2025, Янв 2026, Фев 2026
FORECAST_MONTHS = [(2025, 12), (2026, 1), (2026, 2)]
FORECAST_LABELS = ["Дек 2025", "Янв 2026", "Фев 2026"]

# Обучение: всё до декабря 2025
TRAIN_END_YEAR, TRAIN_END_MONTH = 2025, 11


# ── Загрузка ──────────────────────────────────────────────────────────────────
def load_data(filepath):
    df = pd.read_excel(filepath)
    df_train = df[
        ~((df['Год'] == 2025) & (df['Месяц'] == 12)) &
        ~(df['Год'] == 2026)
    ].copy()
    df_train['date'] = pd.to_datetime(
        df_train['Год'].astype(str) + '-' +
        df_train['Месяц'].astype(str).str.zfill(2) + '-01'
    )
    monthly = (df_train
               .groupby(['date', 'Номер группы'])[['Продажа', 'Ремонт']]
               .sum()
               .reset_index())

    # Факт за Дек 25 / Янв 26 / Фев 26
    actual = {}
    for y, m in FORECAST_MONTHS:
        sub = df[(df['Год'] == y) & (df['Месяц'] == m)]
        for _, row in sub.iterrows():
            g = int(row['Номер группы']) if not pd.isna(row['Номер группы']) else None
            if g is None:
                continue
            if g not in actual:
                actual[g] = {'sale': [0, 0, 0], 'repair': [0, 0, 0]}
            idx = FORECAST_MONTHS.index((y, m))
            actual[g]['sale'][idx]   += float(row['Продажа'] or 0)
            actual[g]['repair'][idx] += float(row['Ремонт']  or 0)

    # Мета
    def parse_analogs(v):
        if pd.isna(v) or str(v).strip() in ('', 'nan'):
            return ''
        try:
            return ', '.join(str(x) for x in ast.literal_eval(str(v)))
        except Exception:
            return str(v).strip("()'\"")

    group_meta = (df.groupby('Номер группы')
                  .agg(Номенклатура=('Номенклатура', 'first'),
                       Артикул=('Артикул', 'first'),
                       ТипТехники=('Тип техники', 'first'),
                       all_analogs=('all_analogs', 'first'))
                  .reset_index())
    group_meta['Аналоги'] = group_meta['all_analogs'].apply(parse_analogs)
    group_meta['Номер группы'] = group_meta['Номер группы'].astype(int)

    all_groups = set(group_meta['Номер группы'].tolist())
    return df_train, monthly, group_meta, actual, all_groups


# ── Обнаружение и замена выбросов ─────────────────────────────────────────────
def remove_outliers(series: pd.Series, iqr_factor: float = 1.5) -> tuple:
    """
    Заменяет выбросы (> Q3 + factor*IQR) на скользящую медиану по 3 соседям.
    Возвращает (очищенный ряд, список выбросов [(дата, исходное, замена)]).
    """
    s = series.copy()
    q1 = s.quantile(0.25)
    q3 = s.quantile(0.75)
    iqr = q3 - q1
    upper = q3 + iqr_factor * iqr

    # Нижний порог не используем (нули — норма для запчастей)
    outlier_mask = s > upper

    outlier_log = []
    for idx in s[outlier_mask].index:
        original = s[idx]
        # Медиана ближайших 3 соседей (без самого выброса)
        pos = s.index.get_loc(idx)
        neighbors = []
        for offset in [-3, -2, -1, 1, 2, 3]:
            ni = pos + offset
            if 0 <= ni < len(s) and not outlier_mask.iloc[ni]:
                neighbors.append(s.iloc[ni])
        replacement = float(np.median(neighbors)) if neighbors else float(s.median())
        s.iloc[pos] = replacement
        outlier_log.append({
            'date':        idx.strftime('%Y-%m'),
            'original':    round(original, 1),
            'replacement': round(replacement, 1),
            'threshold':   round(upper, 1),
        })

    return s, outlier_log


# ── Прогнозирование ─────────────────────────────────────────────────────────
ZERO_THRESHOLD = 0.40   # доля нулей > 40% → Croston

def get_series(monthly, grp, col):
    full_dates = pd.date_range('2023-01-01', '2025-11-01', freq='MS')
    sub = (monthly[monthly['Номер группы'] == grp][['date', col]]
           .set_index('date'))
    return sub[col].reindex(full_dates, fill_value=0).astype(float)


def forecast_croston(series, steps=3):
    """
    Метод Кростона: для прерывистого спроса (много нулей).
    Моделирует отдельно величину заказа и интервал между заказами.
    Прогноз = величина / интервал (постоянный на все шаги).
    """
    s = series.clip(lower=0).values
    alpha = 0.1
    nonzero = [(i, v) for i, v in enumerate(s) if v > 0]
    if not nonzero:
        return pd.Series([0.0] * steps,
                         index=pd.date_range('2025-12-01', periods=steps, freq='MS')), 'Croston'
    z = nonzero[0][1]
    p = float(nonzero[0][0] + 1)
    prev_idx = nonzero[0][0]
    for idx, val in nonzero[1:]:
        interval = idx - prev_idx
        z = alpha * val      + (1 - alpha) * z
        p = alpha * interval + (1 - alpha) * p
        prev_idx = idx
    fc_val = round(z / p, 1) if p > 0 else 0.0
    fc = pd.Series([max(0.0, fc_val)] * steps,
                   index=pd.date_range('2025-12-01', periods=steps, freq='MS'))
    return fc, 'Croston'


def forecast_auto_ets(series, steps=3):
    """
    Перебирает 6 вариантов ETS/Holt-Winters, выбирает лучший по AIC.
    Варианты: тренд (есть/нет) × сезонность (add/mul/нет/полугодие).
    """
    s = series.clip(lower=0)
    has_zeros = (s == 0).any()
    candidates = [
        ('add', 'add',  12, 'HW-add'),
        ('add', 'mul',  12, 'HW-mul'),
        ('add', None,   None, 'Holt (тренд)'),
        (None,  'add',  12,  'ETS (сезон.)'),
        (None,  None,   None, 'Simple ETS'),
        ('add', 'add',  6,   'HW-add (6мес)'),
    ]
    best_fc, best_aic, best_name = None, np.inf, None
    for trend, seasonal, sp, name in candidates:
        if seasonal == 'mul' and has_zeros:
            continue
        if sp and len(s) < 2 * sp:
            continue
        try:
            m = ExponentialSmoothing(s, trend=trend, seasonal=seasonal,
                                     seasonal_periods=sp)
            f = m.fit(optimized=True)
            if f.aic < best_aic:
                best_aic  = f.aic
                best_fc   = f.forecast(steps).clip(lower=0).round(1)
                best_name = name
        except Exception:
            pass
    if best_fc is None:
        v = s.tail(6).mean()
        best_fc = pd.Series([round(v, 1)] * steps,
                            index=pd.date_range('2025-12-01', periods=steps, freq='MS'))
        best_name = 'Mean-6m'
    return best_fc, best_name


def forecast_best(series, steps=3):
    """
    Выбирает метод автоматически:
      > ZERO_THRESHOLD нулей → Croston
      иначе              → auto-ETS (лучший из 6 вариантов по AIC)
    """
    s = series.clip(lower=0)
    zero_ratio = (s == 0).sum() / len(s)
    if zero_ratio > ZERO_THRESHOLD:
        return forecast_croston(s, steps)
    return forecast_auto_ets(s, steps)


def forecast_group(monthly, group_meta, actual, grp):
    meta_row = group_meta[group_meta['Номер группы'] == grp]
    if meta_row.empty:
        return None
    meta = meta_row.iloc[0]
    result = {'grp': grp,
              'name':    str(meta['Номенклатура'])[:70],
              'article': str(meta['Артикул']),
              'type':    str(meta['ТипТехники']),
              'analogs': str(meta['Аналоги']),
              'sale':   {},
              'repair': {}}
    for col, key in [('Продажа', 'sale'), ('Ремонт', 'repair')]:
        raw = get_series(monthly, grp, col)
        zero_ratio = int(round((raw == 0).sum() / len(raw) * 100, 0))
        fc_raw,   method_raw   = forecast_best(raw, 3)
        cleaned, outlier_log   = remove_outliers(raw)
        fc_clean, method_clean = forecast_best(cleaned, 3)
        act_vals = actual.get(grp, {}).get(key, [0, 0, 0])
        result[key] = {
            'fc_raw':       [round(float(fc_raw.iloc[i]),   1) for i in range(3)],
            'fc_clean':     [round(float(fc_clean.iloc[i]), 1) for i in range(3)],
            'act':          [round(float(v), 1) for v in act_vals],
            'method_raw':   method_raw,
            'method_clean': method_clean,
            'outliers':     outlier_log,
            'zero_ratio':   zero_ratio,
        }
    return result

def build_forecast_rows(groups, monthly, group_meta, actual, col_key):
    rows = []
    for grp in groups:
        print(f"    → Группа {grp}...", end='', flush=True)
        r = forecast_group(monthly, group_meta, actual, grp)
        if r:
            rows.append(r)
            d = r[col_key]
            n_out = len(d['outliers'])
            print(f"  {n_out} выбросов | "
                  f"fc_raw={d['fc_raw']} | fc_clean={d['fc_clean']} | act={d['act']}")
        else:
            print(" — нет данных")
    return rows


# ── Excel ─────────────────────────────────────────────────────────────────────
C_DARK = "1A2744"; C_MID = "2D4A8A"; C_ACC = "4A90D9"
C_ORG  = "B8520A"; C_ORL = "FFF3CD"
C_GRN  = "1E6E42"; C_GNL = "E9F7EF"
C_RED  = "922B21"; C_RDL = "FDEDEC"
C_STR  = "EBF3FB"; C_WHT = "FFFFFF"
C_TL   = "117A65"; C_TLL = "D5F5E3"
C_PRP  = "5B2C6F"; C_PRL = "F4ECF7"  # purple for cleaned forecast

def fl(c):  return PatternFill("solid", fgColor=c)
def fn(bold=False, color="000000", size=9, italic=False):
    return Font(name='Arial', bold=bold, color=color, size=size, italic=italic)
_t  = Side(border_style="thin",   color="CCCCCC")
_tb = Side(border_style="medium", color="2D4A8A")
BRD = Border(left=_t, right=_t, top=_t, bottom=_t)
CTR = Alignment(horizontal='center', vertical='center', wrap_text=True)
LFT = Alignment(horizontal='left',   vertical='center', wrap_text=True)
RGT = Alignment(horizontal='right',  vertical='center')

def sc(ws, r, c, v, _fn=None, _fl=None, _al=None, fmt=None):
    cell = ws.cell(row=r, column=c, value=v)
    if _fn: cell.font = _fn
    if _fl: cell.fill = _fl
    if _al: cell.alignment = _al
    if fmt: cell.number_format = fmt
    cell.border = BRD

def mc(ws, r1, c1, r2, c2, v, _fn=None, _fl=None, _al=None):
    ws.merge_cells(start_row=r1, start_column=c1, end_row=r2, end_column=c2)
    cell = ws.cell(row=r1, column=c1, value=v)
    if _fn: cell.font = _fn
    if _fl:
        for row in ws.iter_rows(min_row=r1, max_row=r2, min_col=c1, max_col=c2):
            for c in row: c.fill = _fl; c.border = BRD
    if _al: cell.alignment = _al


# ─────────────────────────────────────────────────────────────────────────────
# COL LAYOUT (per sheet):
#  1=№  2=Гр  3=Артикул  4=Номенклатура  5=Тип  6=Аналоги
#  Per month (×3):
#    col_base + 0 = Прогноз (raw)
#    col_base + 1 = Прогноз (без выбросов)
#    col_base + 2 = Факт
#    col_base + 3 = Δ raw
#    col_base + 4 = Δ clean
#  MONTH 1 cols: 7-11  MONTH 2: 12-16  MONTH 3: 17-21
#  ИТОГО: 22=fc_raw  23=fc_clean  24=Факт  25=Δraw  26=Δclean
# ─────────────────────────────────────────────────────────────────────────────

def build_sheet(ws, rows, col_label, col_key):
    ws.sheet_view.showGridLines = False
    NCOLS = 26

    # Title
    ws.row_dimensions[1].height = 28
    mc(ws, 1, 1, 1, NCOLS,
       f"ПРОГНОЗ vs ФАКТ — {col_label} | Дек 2025 – Фев 2026",
       _fn=fn(True, C_WHT, 13), _fl=fl(C_DARK), _al=CTR)

    ws.row_dimensions[2].height = 14
    mc(ws, 2, 1, 2, NCOLS,
       "Обучение: янв 2023 – ноя 2025 | "
       "Выбросы: IQR × 1.5 → заменяются медианой соседей | "
       "Δ = Факт – Прогноз | Зелёный = модель недооценила, Красный = переоценила",
       _fn=fn(italic=True, color="555555", size=8), _fl=fl("F0F4FA"), _al=CTR)

    # Header rows 4-5
    ws.row_dimensions[4].height = 20
    ws.row_dimensions[5].height = 18

    for c, v in [(1,"№"),(2,"Группа"),(3,"Артикул"),(4,"Номенклатура"),(5,"Тип"),(6,"Аналоги")]:
        mc(ws, 4, c, 5, c, v, _fn=fn(True, C_WHT, 9), _fl=fl(C_DARK), _al=CTR)

    month_colors = ["2C3E50", "1A252F", "0D1B2A"]
    for i, (lbl, bg) in enumerate(zip(FORECAST_LABELS, month_colors)):
        base = 7 + i * 5
        mc(ws, 4, base, 4, base+4, lbl,
           _fn=fn(True, C_WHT, 10), _fl=fl(bg), _al=CTR)
        sub_hdrs = [
            ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
            ("Факт",    C_TL),  ("Δ (осн)",  C_RED),
            ("Δ (чист)",C_GRN),
        ]
        for j, (sh, sc_c) in enumerate(sub_hdrs):
            sc(ws, 5, base+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    mc(ws, 4, 22, 4, 26, "ИТОГО КВАРТАЛ",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_GRN), _al=CTR)
    for j, (sh, sc_c) in enumerate([
        ("Прогноз", C_ORG), ("Без выбр.", C_PRP),
        ("Факт", C_TL), ("Δ (осн)", C_RED), ("Δ (чист)", C_GRN)
    ]):
        sc(ws, 5, 22+j, sh, _fn=fn(True, C_WHT, 8), _fl=fl(sc_c), _al=CTR)

    # Data rows
    for idx, row in enumerate(rows):
        r = 6 + idx
        ws.row_dimensions[r].height = 28
        bg = C_STR if idx % 2 == 0 else C_WHT
        d  = row[col_key]

        sc(ws, r, 1, idx+1,          _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 2, row['grp'],      _fn=fn(True),    _fl=fl(bg), _al=CTR)
        sc(ws, r, 3, row['article'],  _fn=fn(size=8),  _fl=fl(bg), _al=CTR)
        sc(ws, r, 4, row['name'],     _fn=fn(size=8),  _fl=fl(bg), _al=LFT)
        sc(ws, r, 5, row['type'],     _fn=fn(size=8),  _fl=fl(bg), _al=CTR)

        # Аналоги + выбросы
        out_note = ""
        if d['outliers']:
            parts = [f"{o['date']}: {o['original']:.0f}→{o['replacement']:.0f}"
                     for o in d['outliers']]
            out_note = " | ⚠ выбросы: " + "; ".join(parts)
        method_note = f" | Метод: {d['method_clean']}"
        zeros_note  = f" | Нули: {d.get('zero_ratio',0)}%"
        sc(ws, r, 6, row['analogs'] + out_note + method_note + zeros_note,
           _fn=fn(size=7), _fl=fl(bg), _al=LFT)

        sum_raw = 0; sum_clean = 0; sum_act = 0
        for i in range(3):
            base    = 7 + i * 5
            fc_raw  = round(d['fc_raw'][i],   1)
            fc_cln  = round(d['fc_clean'][i], 1)
            act_v   = round(d['act'][i],      1)
            d_raw   = round(act_v - fc_raw,   1)
            d_cln   = round(act_v - fc_cln,   1)
            sum_raw   += fc_raw
            sum_clean += fc_cln
            sum_act   += act_v

            sc(ws, r, base,   fc_raw, _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+1, fc_cln, _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
            sc(ws, r, base+2, act_v,  _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')

            for delta, col_i in [(d_raw, base+3), (d_cln, base+4)]:
                clr = C_TL  if delta >= 0 else C_RED
                bg2 = C_TLL if delta >= 0 else C_RDL
                sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
                   _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

        # Totals
        td_raw  = round(sum_act - sum_raw,   1)
        td_cln  = round(sum_act - sum_clean, 1)
        sc(ws, r, 22, round(sum_raw,   1), _fn=fn(True, C_ORG), _fl=fl(C_ORL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 23, round(sum_clean, 1), _fn=fn(True, C_PRP), _fl=fl(C_PRL), _al=RGT, fmt='#,##0.0')
        sc(ws, r, 24, round(sum_act,   1), _fn=fn(True, C_TL),  _fl=fl(C_TLL), _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(td_raw, 25), (td_cln, 26)]:
            clr = C_TL  if delta >= 0 else C_RED
            bg2 = C_TLL if delta >= 0 else C_RDL
            sc(ws, r, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True, clr), _fl=fl(bg2), _al=CTR)

    # Grand total row
    tr = 6 + len(rows)
    ws.row_dimensions[tr].height = 20
    mc(ws, tr, 1, tr, 6, "ИТОГО ТОП-10",
       _fn=fn(True, C_WHT, 10), _fl=fl(C_DARK), _al=CTR)

    for i in range(3):
        base = 7 + i * 5
        sf = round(sum(row[col_key]['fc_raw'][i]   for row in rows), 1)
        sc2= round(sum(row[col_key]['fc_clean'][i] for row in rows), 1)
        sa = round(sum(row[col_key]['act'][i]      for row in rows), 1)
        sc(ws, tr, base,   sf,  _fn=fn(True,C_WHT), _fl=fl(C_ORG), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+1, sc2, _fn=fn(True,C_WHT), _fl=fl(C_PRP), _al=RGT, fmt='#,##0.0')
        sc(ws, tr, base+2, sa,  _fn=fn(True,C_WHT), _fl=fl(C_TL),  _al=RGT, fmt='#,##0.0')
        for delta, col_i in [(round(sa-sf,1), base+3), (round(sa-sc2,1), base+4)]:
            sc(ws, tr, col_i, f"{'+' if delta>=0 else ''}{delta:.1f}",
               _fn=fn(True,C_WHT), _fl=fl(C_RED if delta<0 else C_TL), _al=CTR)

    tf  = round(sum(sum(row[col_key]['fc_raw'])   for row in rows), 1)
    tc  = round(sum(sum(row[col_key]['fc_clean']) for row in rows), 1)
    ta  = round(sum(sum(row[col_key]['act'])      for row in rows), 1)
    sc(ws,tr,22,tf,_fn=fn(True,C_WHT,10),_fl=fl(C_ORG),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,23,tc,_fn=fn(True,C_WHT,10),_fl=fl(C_PRP),_al=RGT,fmt='#,##0.0')
    sc(ws,tr,24,ta,_fn=fn(True,C_WHT,10),_fl=fl(C_TL), _al=RGT,fmt='#,##0.0')
    for delta, col_i in [(round(ta-tf,1),25),(round(ta-tc,1),26)]:
        sc(ws,tr,col_i,f"{'+' if delta>=0 else ''}{delta:.1f}",
           _fn=fn(True,C_WHT,10),_fl=fl(C_RED if delta<0 else C_TL),_al=CTR)

    # MAPE row
    mr = tr + 1
    ws.row_dimensions[mr].height = 16
    mape_raw  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_raw'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mape_cln  = np.mean([abs(sum(row[col_key]['act'][i] - row[col_key]['fc_clean'][i]
                               for i in range(3)) /
                            (sum(row[col_key]['act']) + 1e-9)) * 100
                         for row in rows])
    mc(ws, mr, 1, mr, NCOLS,
       f"MAPE: Прогноз (осн.) = {mape_raw:.1f}%  |  "
       f"Прогноз (без выбросов) = {mape_cln:.1f}%  |  "
       f"{'✅ Очистка выбросов улучшила точность' if mape_cln < mape_raw else '⚠ Очистка не улучшила'}  |  "
       f"Методы: auto-ETS (6 вариантов) + Croston (при >40% нулей)",
       _fn=fn(bold=True, color=C_GRN if mape_cln < mape_raw else C_RED, size=9),
       _fl=fl("FDFEFE"), _al=LFT)

    # Column widths
    ws.column_dimensions['A'].width = 4
    ws.column_dimensions['B'].width = 7
    ws.column_dimensions['C'].width = 13
    ws.column_dimensions['D'].width = 34
    ws.column_dimensions['E'].width = 13
    ws.column_dimensions['F'].width = 40
    for i in range(3):
        for j, w in enumerate([10, 11, 10, 8, 8]):
            ws.column_dimensions[get_column_letter(7 + i*5 + j)].width = w
    for j, w in enumerate([10, 11, 10, 8, 8]):
        ws.column_dimensions[get_column_letter(22 + j)].width = w
    ws.freeze_panes = 'G6'


def build_outlier_sheet(ws, rows_sale, rows_rep):
    """Отдельный лист — подробная таблица всех выбросов."""
    ws.title = "⚠ Выбросы"
    ws.sheet_view.showGridLines = False

    ws.row_dimensions[1].height = 26
    mc(ws, 1, 1, 1, 8,
       "ОБНАРУЖЕННЫЕ ВЫБРОСЫ — автоматическая корректировка (IQR × 1.5)",
       _fn=fn(True, C_WHT, 12), _fl=fl(C_DARK), _al=CTR)

    hdrs = ["Группа", "Номенклатура", "Тип (прод/рем)",
            "Дата", "Исходное значение", "Порог IQR",
            "Заменено на", "Δ корректировка"]
    ws.row_dimensions[3].height = 18
    for ci, h in enumerate(hdrs):
        sc(ws, 3, 1+ci, h, _fn=fn(True, C_WHT, 9), _fl=fl(C_MID), _al=CTR)

    r = 4
    for rows, kind in [(rows_sale, "Продажи"), (rows_rep, "Ремонт")]:
        for row in rows:
            for o in row['sale' if kind == "Продажи" else 'repair']['outliers']:
                ws.row_dimensions[r].height = 16
                delta = round(o['replacement'] - o['original'], 1)
                bg = C_RDL
                sc(ws,r,1,row['grp'],          _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,2,row['name'][:50],    _fn=fn(size=8),_fl=fl(bg),_al=LFT)
                sc(ws,r,3,kind,                _fn=fn(size=8),_fl=fl(bg),_al=CTR)
                sc(ws,r,4,o['date'],           _fn=fn(True),  _fl=fl(bg),_al=CTR)
                sc(ws,r,5,o['original'],       _fn=fn(True,C_RED),_fl=fl(C_RDL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,6,o['threshold'],      _fn=fn(),      _fl=fl(bg),_al=RGT,fmt='#,##0.0')
                sc(ws,r,7,o['replacement'],    _fn=fn(True,C_GRN),_fl=fl(C_GNL),_al=RGT,fmt='#,##0.0')
                sc(ws,r,8,f"{delta:+.1f}",     _fn=fn(True,C_RED),_fl=fl(bg),_al=CTR)
                r += 1

    for ci, w in enumerate([7,40,12,10,14,12,13,14]):
        ws.column_dimensions[get_column_letter(1+ci)].width = w
    ws.freeze_panes = 'A4'


# ── Точность прогноза (текстовый вывод) ──────────────────────────────────────
def print_accuracy(rows, col_key, label):
    print(f"\n  {'─'*75}")
    print(f"  ТОЧНОСТЬ — {label}")
    print(f"  {'Гр.':>5}  {'Нули':>5}  {'Метод':>20}  {'fc_raw Q3':>10}  "
          f"{'fc_cln Q3':>10}  {'Факт Q3':>9}  {'Δ raw':>7}  {'Δ clean':>8}  MAPE_raw  MAPE_cln")
    print(f"  {'─'*75}")
    for row in rows:
        d = row[col_key]
        fr = round(sum(d['fc_raw']),   1)
        fc = round(sum(d['fc_clean']), 1)
        fa = round(sum(d['act']),      1)
        dr = round(fa - fr, 1)
        dc = round(fa - fc, 1)
        mape_r = abs(dr) / (fa + 1e-9) * 100
        mape_c = abs(dc) / (fa + 1e-9) * 100
        better = "✅" if mape_c < mape_r else "  "
        method = d.get('method_clean', '—')
        zeros  = d.get('zero_ratio', 0)
        print(f"  {row['grp']:>5}  {zeros:>4}%  {method:>20}  {fr:>10.1f}  "
              f"{fc:>10.1f}  {fa:>9.1f}  {dr:>+7.1f}  {dc:>+8.1f}  "
              f"{mape_r:>7.1f}%  {mape_c:>7.1f}% {better}")


# ── MAIN ──────────────────────────────────────────────────────────────────────
def main():
    try:
        script_dir = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        script_dir = os.getcwd()

    data_path = os.path.join(script_dir, DATA_FILE)
    if not os.path.exists(data_path):
        print(f"\n❌  Файл не найден: {data_path}")
        sys.exit(1)

    print("\n" + "="*65)
    print("  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов")
    print("="*65)
    print(f"\n  Загрузка {DATA_FILE}...")
    df_train, monthly, group_meta, actual, all_groups = load_data(data_path)
    print(f"  Загружено. Групп: {len(all_groups)}")

    # ── Топ-20 для справки ────────────────────────────────────────────────────
    summary = (df_train.groupby('Номер группы')
               .agg(Продажа=('Продажа', 'sum'),
                    Ремонт=('Ремонт', 'sum'),
                    Номенклатура=('Номенклатура', 'first'))
               .reset_index()
               .sort_values('Продажа', ascending=False))

    print("\n  Топ-20 по продажам (для справки):")
    for _, row in summary.head(20).iterrows():
        print(f"    Гр.{int(row['Номер группы']):4d} | "
              f"Прод: {row['Продажа']:6.0f} | "
              f"Рем: {row['Ремонт']:5.0f} | "
              f"{str(row['Номенклатура'])[:50]}")

    def parse_input(raw):
        nums = set()
        raw = raw.replace(',', ' ')
        for part in raw.split():
            if '-' in part and not part.startswith('-'):
                a, b = part.split('-', 1)
                try: nums.update(range(int(a), int(b) + 1))
                except ValueError: pass
            else:
                try: nums.add(int(part))
                except ValueError: pass
        return sorted(nums)

    # ── Интерактивный цикл ────────────────────────────────────────────────────
    while True:
        print("\n" + "-"*65)
        print("  Форматы: '9 146 205'  |  '9,146'  |  '9-15'  |  'топ10'")
        print("  'поиск <слово>' — поиск по названию  |  'выход' — выйти")
        raw = input("\n  Номера групп: ").strip()

        if raw.lower() in ('выход', 'exit', 'q'):
            print("  До свидания!"); return

        if raw.lower().startswith('поиск'):
            q = raw[5:].strip() or input("  Слово для поиска: ").strip()
            found = summary[summary['Номенклатура'].str.contains(q, case=False, na=False)]
            if found.empty:
                print(f"  Ничего не найдено по: '{q}'")
            else:
                for _, r in found.iterrows():
                    print(f"    Гр.{int(r['Номер группы']):4d} | "
                          f"{r['Продажа']:6.0f} прод. | "
                          f"{str(r['Номенклатура'])[:55]}")
            continue

        if raw.lower() in ('топ10', 'топ', 'top'):
            requested = summary.head(10)['Номер группы'].astype(int).tolist()
        else:
            requested = parse_input(raw)

        if not requested:
            print("  ⚠️  Не удалось распознать номера. Попробуйте ещё раз.")
            continue

        valid   = [g for g in requested if g in all_groups]
        invalid = [g for g in requested if g not in all_groups]
        if invalid:
            print(f"  ⚠️  Не найдены в данных: {invalid}")
        if not valid:
            print("  ❌  Ни одна группа не найдена.")
            continue

        top10_sale = valid
        top10_rep  = valid
        print(f"\n  Выбранные группы: {valid}")

        print("\n  Строим прогноз...")
        all_results = {}
        for grp in list(dict.fromkeys(top10_sale + top10_rep)):
            print(f"    → Группа {grp}...", end='', flush=True)
            r = forecast_group(monthly, group_meta, actual, grp)
            if r:
                all_results[grp] = r
                d = r['sale']
                print(f"  {len(d['outliers'])} выбросов | "
                      f"fc={d['fc_raw']} → clean={d['fc_clean']} | fact={d['act']}")
            else:
                print("  нет данных")

        rows_sale = [all_results[g] for g in top10_sale if g in all_results]
        rows_rep  = [all_results[g] for g in top10_rep  if g in all_results]

        print_accuracy(rows_sale, 'sale',   'ПРОДАЖИ')
        print_accuracy(rows_rep,  'repair', 'РЕМОНТ')

        wb = openpyxl.Workbook()
        ws1 = wb.active; ws1.title = "📈 Продажи"
        build_sheet(ws1, rows_sale, "ПРОДАЖИ", "sale")
        ws2 = wb.create_sheet("🔧 Ремонт")
        build_sheet(ws2, rows_rep, "РЕМОНТ", "repair")
        ws3 = wb.create_sheet()
        build_outlier_sheet(ws3, rows_sale, rows_rep)

        try:
            _sdir = os.path.dirname(os.path.abspath(__file__))
        except NameError:
            _sdir = os.getcwd()
        groups_str = '_'.join(str(g) for g in valid[:5])
        out_path = os.path.join(_sdir, f"forecast_vs_fact_{groups_str}.xlsx")
        wb.save(out_path)
        print(f"\n  ✅  Файл сохранён: {out_path}")
        print("  Листы: 📈 Продажи | 🔧 Ремонт | ⚠ Выбросы")

        again = input("\n  Ещё один прогноз? (да/нет): ").strip().lower()
        if again not in ('да', 'д', 'y', 'yes'):
            print("  До свидания!"); return


if __name__ == "__main__":
    main()


  ПРОГНОЗ vs ФАКТ | Дек 2025 – Фев 2026 | С обработкой выбросов

  Загрузка final.xlsx...
  Загружено. Групп: 2239

  Топ-20 по продажам (для справки):
    Гр.   9 | Прод:   3123 | Рем:   539 | Замок зажигания с ключом IHP-30 (1020520943) 10205
    Гр. 226 | Прод:   1757 | Рем:   108 | Фильтр гидравлический напорный 00004697
    Гр. 205 | Прод:   1280 | Рем:   182 | Фильтр гидравлический D0030A10NHA
    Гр. 146 | Прод:   1262 | Рем:   383 | Фильтр топливный тонкой очистки (резьбовой) ЕВРО-3
    Гр. 116 | Прод:   1193 | Рем:   187 | Фильтр масляный MD162326-V
    Гр. 147 | Прод:   1103 | Рем:   410 | Фильтр воздушный внешний DIFA 43121 + внутренний D
    Гр. 144 | Прод:   1073 | Рем:   353 | Фильтр топливный 90031462A
    Гр. 118 | Прод:   1008 | Рем:   163 | Фильтр масляный 140517050
    Гр.   7 | Прод:    959 | Рем:   174 | Кнопка Стоп 122514GT
    Гр. 113 | Прод:    955 | Рем:   382 | Фильтр воздушный внешний DIFA 43121 + внутренний D
    Гр. 332 | Прод:    771 | Рем:   140 | Кабель